# IEEE 57-bus + High-Speed-Rail Coupled Benchmark

Companion data-generation notebook for

> A. Arif, "AC-Guided Co-Expansion Planning of Power Grids Hosting High-Speed Railways," *IEEE Access*, 2026 (under review).

Repository: https://github.com/Anmar001/ieee57-hsr-benchmark

The notebook builds, from public data and declared assumptions, every input that the planning and validation stages
of the paper consume.

1. The IEEE 57-bus host grid with schematic coordinates and a 400 km geographic scale.
2. Six synthetic 25 kV traction corridors, their stations and 29 traction substations (TSS), placed at existing
   buses or on new line taps.
3. The expanded MATPOWER case of the host grid after the tap splits (`case57_hsr.m`, 63 buses, 86 branches).
4. A calibrated CR400AF-class trainset and a one-second longitudinal simulation of the cyclic nominal timetable,
   giving the per-substation demand and surplus-regeneration signature (`hsr_demand_signature.npz`).
5. Pre-expansion AC diagnostics of the host grid under the nominal traction demand (Figs. 4 and 5 of the paper).
6. The 15-minute planning envelopes of the nominal day and the seeded two-regime timetable-jitter generator, with the
   seed families used for training and validation in the paper (Section 10).

Everything is deterministic. Each jittered day depends only on its seed, so every validation day of the paper can be
regenerated here to the second.

**Not included.** The optimization pipeline of the paper (the AMPL master problem, the exact-AC subproblem and
validator, and the guidance loop) is not part of this release. It requires an AMPL licence with the Gurobi and Knitro
solvers and is available from the author on reasonable request. Nothing in this notebook needs a solver.

**Requirements.** Python 3.10 or later with `numpy`, `matplotlib` and `pypower`. The notebook runs top to bottom in a
few minutes. Each 100-day validation family exported in Section 10.3 adds one to two minutes.

**Outputs** are written to the working directory and collected under `benchmark/` by the collection cell (Section 11).

**Citation.** If you use this benchmark, please cite the paper above.

Numbers in square brackets in the code comments, such as `[23]`, refer to the bibliography of the paper. The last
cell lists these references, the one external data source (PGLib-OPF), and the parameters that are declared
assumptions without a cited source.


In [ ]:
# Dependencies (install once if missing):  pip install numpy matplotlib pypower
import sys
from importlib.metadata import version
print("python", sys.version.split()[0], "| numpy", version("numpy"), "| matplotlib", version("matplotlib"),
      "| pypower", version("PYPOWER"))


## Overview of the build

The notebook embeds the IEEE 57-bus transmission grid in a 400 km geography, overlays a synthetic high-speed-rail
(HSR) network, and simulates a full day of train movements to obtain the per-substation demand and regeneration
signature that drives the planning model of the paper.

| Section | Content |
|---|---|
| 1 | schematic coordinates of the IEEE 57-bus grid |
| 2 | six 2x25 kV AC traction corridors routed over grid buses |
| 3 | the 400 km geographic scale and geometry checks |
| 4 | drawing of the coupled network |
| 5 | station and TSS placement (TSS at existing buses where close enough, else on new line taps) |
| 6 | impedance split of the tapped lines and the expanded MATPOWER case `case57_hsr` |
| 7 | one-second train-movement simulation and the per-TSS demand and surplus-regeneration signature |
| 8 | verification tables and figures |
| 9 | pre-expansion power-flow and AC-OPF diagnostics of the host grid |
| 10 | 15-minute planning envelopes and the seeded timetable-jitter days |
| 11 | collection of the benchmark files |

The modelling target is modern Chinese high-speed rail (350 km/h, 2x25 kV autotransformer feeding). The grid keeps the
standard IEEE 57-bus per-unit impedances. The geography informs only the rail overlay and the newly added elements.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from collections import OrderedDict, defaultdict

# pypower ships code that imports numpy.in1d, removed in numpy >= 2.0. Shim it.
if not hasattr(np, "in1d"):
    np.in1d = np.isin
from pypower.api import case57

plt.rcParams["figure.dpi"] = 110

## 1. Schematic coordinates for IEEE-57

Unit square `[0,1] x [0,1]`. These positions are an internally consistent abstract
embedding (topology-preserving), later given physical meaning by the 400 km scale.

In [ ]:
# Synthetic, topology-preserving schematic coordinates for the IEEE-57 buses.
# The IEEE-57 case ships with no geographic data and no base kV [21]; these coordinates
# are assigned for visualisation and the HSR overlay only (Sections 2 to 5).
pos57 = {
    1:(0.72,0.92),  2:(0.36,0.95),  3:(0.24,0.78),  4:(0.23,0.67),  5:(0.06,0.62),
    6:(0.06,0.50),  7:(0.18,0.36),  8:(0.45,0.05),  9:(0.70,0.13), 10:(0.85,0.40),
   11:(0.74,0.50), 12:(0.96,0.55), 13:(0.65,0.54), 14:(0.57,0.58), 15:(0.46,0.62),
   16:(0.85,0.66), 17:(0.70,0.74), 18:(0.25,0.62), 19:(0.29,0.56), 20:(0.36,0.54),
   21:(0.38,0.49), 22:(0.43,0.43), 23:(0.34,0.44), 24:(0.31,0.40), 25:(0.33,0.37),
   26:(0.28,0.47), 27:(0.24,0.43), 28:(0.22,0.38), 29:(0.17,0.30), 30:(0.32,0.30),
   31:(0.35,0.28), 32:(0.46,0.28), 33:(0.55,0.26), 34:(0.43,0.33), 35:(0.41,0.35),
   36:(0.43,0.39), 37:(0.49,0.41), 38:(0.45,0.46), 39:(0.52,0.43), 40:(0.52,0.39),
   41:(0.58,0.36), 42:(0.58,0.40), 43:(0.68,0.36), 44:(0.52,0.52), 45:(0.52,0.56),
   46:(0.58,0.55), 47:(0.60,0.44), 48:(0.64,0.48), 49:(0.69,0.32), 50:(0.62,0.25),
   51:(0.91,0.32), 52:(0.26,0.26), 53:(0.34,0.18), 54:(0.48,0.20), 55:(0.55,0.17),
   56:(0.52,0.36), 57:(0.54,0.45),
}

## 2. HSR traction corridors

Each corridor is a polyline through IEEE 57-bus buses. The polylines are the rail alignments. Stations and traction
substations are placed along them in Section 5.

**Waypoint buses.** Two corridors would otherwise contain legs longer than 120 km, whose mid-leg substations would need
long grid ties. Four grid buses that lie close to those alignments are therefore used as intermediate waypoints.

- **C1 (NE):** `2-1-12-9` becomes `2-1-16-12-10-9` (bus 16 lies about 10 km from the 1-12 line, bus 10 about 5 km from 12-9).
- **C3 (South):** `6-7-8-9` becomes `6-7-52-53-8-9` (buses 52 and 53 lie within 2 km of the 7-8 line).

The waypoints also add points at which traction demand and regenerative braking interact with the grid.


In [ ]:
# Synthetic HSR overlay: six 25 kV AC traction corridors routed over IEEE-57 buses.
# Corridor shapes and lengths are declared assumptions at trunk high-speed scale; the waypoint
# buses 16, 10, 52 and 53 keep every leg below about 120 km (Section 2).
corridors = OrderedDict({
    "HSR Corridor 1 (NE)":          [2, 1, 16, 12, 10, 9],
    "HSR Corridor 2 (West)":        [6, 4, 3],
    "HSR Corridor 3 (South)":       [6, 7, 52, 53, 8, 9],
    "HSR Corridor 4 (North spine)": [22, 18, 4, 3, 2],
    "HSR Corridor 5 (Mid E-W)":     [12, 11, 22, 23, 6],
    "HSR Corridor 6 (Central N-S)": [1, 15, 22, 8],
})

corridor_colors = {
    "HSR Corridor 1 (NE)":          "#1f77b4",
    "HSR Corridor 2 (West)":        "#ff7f0e",
    "HSR Corridor 3 (South)":       "#2ca02c",
    "HSR Corridor 4 (North spine)": "#d62728",
    "HSR Corridor 5 (Mid E-W)":     "#9467bd",
    "HSR Corridor 6 (Central N-S)": "#8c564b",
}

new_gen_candidates = [5, 12, 17, 18, 51, 26, 15, 53]

# --- Load IEEE-57 ---
mpc          = case57()
bus0         = mpc["bus"].copy()
branch0      = mpc["branch"].copy()
gen0_full    = mpc["gen"].copy()
gencost0_full= mpc["gencost"].copy()
baseMVA      = float(mpc["baseMVA"])
grid_buses   = bus0[:, 0].astype(int).tolist()

# Branch thermal (MVA) ratings of the PGLib-OPF case pglib_opf_case57_ieee (v23.07) [PGLib].
# The stock case57 leaves the ratings at 0 (unlimited); explicit ratings make congestion meaningful.
pglib_rateA = [1005, 326, 767, 201, 191, 283, 167, 570, 171, 331, 98, 178, 647, 323, 317,
               140, 266, 530, 53, 69, 414, 405, 228, 384, 484, 353, 160, 512, 36, 57, 38,
               213, 1617, 97, 25, 24, 621, 97, 259, 408, 453, 121, 50, 33, 552, 31, 313,
               427, 629, 245, 655, 530, 834, 40, 72, 72, 450, 282, 400, 409, 993, 191, 195,
               113, 412, 154, 125, 236, 99, 103, 192, 212, 25, 38, 72, 22, 94, 139, 511, 244]
assert len(pglib_rateA) == branch0.shape[0]
branch0[:, 5] = pglib_rateA

# --- Coupling-bus bookkeeping ---
counts = {}
for _, buses_ in corridors.items():
    for b in buses_:
        counts[b] = counts.get(b, 0) + 1
coupling_buses     = sorted(counts.keys())
intersection_buses = sorted([b for b, c in counts.items() if c >= 2])

# --- Which corridors share each undirected segment (for transit-map offsets) ---
corridor_order = list(corridors.keys())
seg_corridors  = defaultdict(list)
for name, cbuses in corridors.items():
    for i in range(len(cbuses) - 1):
        key = tuple(sorted((cbuses[i], cbuses[i + 1])))
        if name not in seg_corridors[key]:
            seg_corridors[key].append(name)

print(f"Grid buses: {len(grid_buses)} | corridor waypoint buses: {len(coupling_buses)} | "
      f"Intersections: {intersection_buses}")

## 3. Geographic scale (400 km)

One scale for the whole embedding. `SPAN_KM` is the corner-to-corner (bounding-box
diagonal) distance; everything else derives from it.

In [ ]:
# Full-network span of 400 km (declared assumption). It gives corridors of about 110-410 km, the scale of
# trunk high-speed lines, and keeps the peak traction load a large but not overwhelming share of the grid.
SPAN_KM = 400.0
xs = [p[0] for p in pos57.values()]
ys = [p[1] for p in pos57.values()]
DIAG_UNITS = np.hypot(max(xs) - min(xs), max(ys) - min(ys))
U2KM = SPAN_KM / DIAG_UNITS            # unit -> km conversion
TSS_SPACING_KM = 50.0                  # target TSS spacing for 2x25 kV feeding (declared assumption)

def seg_km(a, b):
    return float(np.hypot(pos57[a][0] - pos57[b][0],
                          pos57[a][1] - pos57[b][1])) * U2KM

def turn_deg(a, b, c):
    v1 = np.array(pos57[b]) - np.array(pos57[a])
    v2 = np.array(pos57[c]) - np.array(pos57[b])
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    if n1 == 0 or n2 == 0:
        return 0.0
    return float(np.degrees(np.arccos(np.clip(np.dot(v1, v2) / (n1 * n2), -1, 1))))

print(f"Scale: {SPAN_KM:.0f} km diagonal  ->  1 unit = {U2KM:.1f} km")

### 3a. Corridor lengths and geometry check

For each corridor the table reports the length, the detour ratio against the straight end-to-end distance, the largest
bend between consecutive schematic legs, and the approximate number of TSS at 50 km spacing. The bends are angles
between straight schematic legs at waypoint buses. A real alignment would round them with large-radius curves.


In [ ]:
print(f"{'corridor':<30}{'len(km)':>8}{'legs (km)':>26}")
print("-" * 66)
for name, cb in corridors.items():
    segs  = [seg_km(cb[i], cb[i+1]) for i in range(len(cb) - 1)]
    turns = [turn_deg(cb[i], cb[i+1], cb[i+2]) for i in range(len(cb) - 2)]
    total = sum(segs)
    endpt = seg_km(cb[0], cb[-1])
    detour = total / endpt if endpt else float("nan")
    leglbl = "+".join(f"{s:.0f}" for s in segs)
    print(f"{name:<30}{total:>8.0f}   {leglbl:>23}")
    print(f"{'':<30}  detour {detour:.2f}x | max turn "
          f"{(max(turns) if turns else 0):.0f} deg | "
          f"~{int(sum(round(s / TSS_SPACING_KM) for s in segs))} TSS @50km")

### 3b. IEEE-57 transmission-line lengths at this scale

A sanity check that the grid reads as a plausible regional transmission network
under the chosen scale.

In [ ]:
blen = np.array([seg_km(int(f), int(t)) for f, t in branch0[:, 0:2].astype(int)])
print(f"IEEE-57 line lengths (km): min {blen.min():.0f} | median {np.median(blen):.0f} "
      f"| mean {blen.mean():.0f} | max {blen.max():.0f}")
print(f"lines > 150 km: {(blen > 150).sum()} / {len(blen)}    "
      f"lines > 200 km: {(blen > 200).sum()} / {len(blen)}")

## 4. Draw both networks

Left: full coupled system (IEEE-57 grid + HSR corridors + new-generator candidates),
with a 100 km scale bar. Shared corridor segments are drawn with small perpendicular
offsets, transit-map style. Right: the plain IEEE-57 grid with bus labels.

In [ ]:
grid_segs = [(pos57[int(f)], pos57[int(t)]) for f, t in branch0[:, 0:2].astype(int)]
xy = np.array([pos57[n] for n in grid_buses])

fig, (ax_full, ax_ieee) = plt.subplots(1, 2, figsize=(20, 9))

# ---------- LEFT: full coupled network ----------
ax_full.add_collection(LineCollection(grid_segs, linewidths=1.6, colors="0.70",
                                      alpha=0.85, zorder=1))
ax_full.scatter(xy[:, 0], xy[:, 1], s=18, c="k", alpha=0.9, zorder=2,
                label="IEEE-57 buses")

cb_xy = np.array([pos57[b] for b in coupling_buses])
ax_full.scatter(cb_xy[:, 0], cb_xy[:, 1], s=90, marker="s", facecolors="none",
                edgecolors="0.15", linewidths=2.0, zorder=4,
                label="Corridor waypoint buses")

cand_xy = np.array([pos57[b] for b in new_gen_candidates])
ax_full.scatter(cand_xy[:, 0], cand_xy[:, 1], s=170, marker="*", c="gold",
                edgecolors="0.1", linewidths=0.8, zorder=10,
                label="New gen candidates")
for b in new_gen_candidates:
    x, y = pos57[b]
    ax_full.text(x + 0.012, y + 0.012, f"G{b}", fontsize=9, weight="bold",
                 ha="left", va="bottom", zorder=11)

# corridors with perpendicular offsets on shared segments
offset_spacing = 0.010
corridor_lw    = 3.0
for name, cbuses in corridors.items():
    for i in range(len(cbuses) - 1):
        a, b = cbuses[i], cbuses[i + 1]
        p1 = np.array(pos57[a], float)
        p2 = np.array(pos57[b], float)
        key = tuple(sorted((a, b)))
        sharing = [n for n in corridor_order if n in seg_corridors[key]]
        idx, n = sharing.index(name), len(sharing)
        d = p2 - p1
        L = np.hypot(*d)
        perp = np.array([-d[1], d[0]]) / L if L else np.zeros(2)
        off = (idx - (n - 1) / 2.0) * offset_spacing
        q1, q2 = p1 + perp * off, p2 + perp * off
        ax_full.plot([q1[0], q2[0]], [q1[1], q2[1]], c=corridor_colors[name],
                     lw=corridor_lw, solid_capstyle="round", alpha=0.95, zorder=7,
                     label=name if i == 0 else "")

# 100 km scale bar
bar_km = 100.0
bar_u  = bar_km / U2KM
x0, y0 = min(xs), min(ys) - 0.04
ax_full.plot([x0, x0 + bar_u], [y0, y0], c="k", lw=3, zorder=12)
ax_full.text(x0 + bar_u / 2, y0 - 0.02, f"{bar_km:.0f} km", ha="center",
             va="top", fontsize=9)

ax_full.set_title(f"IEEE-57 + HSR traction corridors  (span {SPAN_KM:.0f} km, "
                  f"1 unit = {U2KM:.0f} km)")
ax_full.axis("off")
ax_full.set_aspect("equal")
handles, labels = ax_full.get_legend_handles_labels()
uniq = {}
for h, l in zip(handles, labels):
    uniq.setdefault(l, h)
ax_full.legend(list(uniq.values()), list(uniq.keys()), loc="upper left", fontsize=9)

# ---------- RIGHT: plain IEEE-57 ----------
ax_ieee.add_collection(LineCollection(grid_segs, linewidths=1.6, colors="0.40",
                                      alpha=0.9, zorder=1))
ax_ieee.scatter(xy[:, 0], xy[:, 1], s=28, c="k", alpha=0.95, zorder=2,
                label="IEEE-57 buses")
for n in grid_buses:
    x, y = pos57[n]
    ax_ieee.text(x + 0.008, y + 0.008, str(n), fontsize=7, ha="left",
                 va="bottom", zorder=3)
ax_ieee.set_title("IEEE 57-bus system")
ax_ieee.axis("off")
ax_ieee.set_aspect("equal")
ax_ieee.legend(loc="upper left", fontsize=9)

plt.tight_layout()
plt.show()

## 5. Train stations and traction substations (TSS)

Both are placed **along the corridor polylines**, parametrised by cumulative distance (km).

**Stations** (ridership points): spaced ~45 km (modern-HSR 30-60 km band), terminals at
corridor ends; near-coincident stops on shared/overlapping segments are merged.

**TSS** (grid feeds): targeted ~50 km along each corridor. Each target is **snapped to the
nearest IEEE-57 bus if within `SNAP_KM`**; otherwise the nearest transmission line is
**tapped and split with a new bus** at the tap point. Duplicates on shared segments merged.

Knobs: `STATION_SPACING_KM`, `TSS_SPACING_KM`, `SNAP_KM` (lower it to trade long ties for
more tapped buses), `DEDUP_KM`.

In [ ]:
STATION_SPACING_KM = 45.0   # target station spacing (declared assumption)
# TSS_SPACING_KM defined above (50 km); SNAP_KM = use existing bus if within this many km
SNAP_KM  = 25.0
DEDUP_KM = 8.0

def _P(b):
    return np.array(pos57[b], float)

def corridor_cumkm(cb):
    c = [0.0]
    for i in range(len(cb) - 1):
        c.append(c[-1] + np.linalg.norm(_P(cb[i+1]) - _P(cb[i])) * U2KM)
    return np.array(c)

def point_at_km(cb, c, d):
    d = min(max(d, 0), c[-1])
    i = int(np.searchsorted(c, d) - 1); i = max(0, min(i, len(cb) - 2))
    seg = c[i+1] - c[i]; t = 0.0 if seg == 0 else (d - c[i]) / seg
    return _P(cb[i]) + t * (_P(cb[i+1]) - _P(cb[i]))

def resample_corridor(cb, spacing_km):
    c = corridor_cumkm(cb); L = c[-1]; n = max(1, int(round(L / spacing_km)))
    return [point_at_km(cb, c, k * L / n) for k in range(n + 1)], L

def dedupe_points(points, tol_km):
    keep = []
    for p in points:
        if all(np.linalg.norm(p - q) * U2KM > tol_km for q in keep):
            keep.append(p)
    return keep

In [ ]:
# ---- STATIONS ----
_raw = []
for name, cb in corridors.items():
    pts, L = resample_corridor(cb, STATION_SPACING_KM)
    _raw += pts
station_xy = dedupe_points(_raw, DEDUP_KM)
stations = {f"ST{i+1:02d}": tuple(np.round(p, 4)) for i, p in enumerate(station_xy)}
print(f"Stations placed: {len(stations)}  (~{STATION_SPACING_KM:.0f} km target spacing)")

In [ ]:
# ---- TSS: snap-to-nearest-bus, else tap-and-split a line ----
def nearest_bus(p):
    ds = {b: np.linalg.norm(p - _P(b)) * U2KM for b in grid_buses}
    b = min(ds, key=ds.get); return b, ds[b]

def nearest_branch_tap(p):
    best = None
    for r in branch0:
        f, t = int(r[0]), int(r[1]); A, B = _P(f), _P(t); AB = B - A
        L2 = float(np.dot(AB, AB))
        if L2 == 0: continue
        tt = float(np.clip(np.dot(p - A, AB) / L2, 0, 1))
        q = A + tt * AB; d = np.linalg.norm(p - q) * U2KM
        if best is None or d < best[3]:
            best = (f, t, tt, d, q)
    return best

# Resample targets on every corridor while retaining corridor membership. Targets on shared
# segments are merged geometrically, but their corridor labels are unioned. This metadata is later
# used to prevent a section near a geometric crossing from being assigned to another corridor's TSS.
_targets = []
for name, cb in corridors.items():
    pts, _L = resample_corridor(cb, TSS_SPACING_KM)
    for p in pts:
        hit = next((x for x in _targets
                    if np.linalg.norm(np.asarray(x["xy"]) - np.asarray(p)) * U2KM <= DEDUP_KM), None)
        if hit is None:
            _targets.append(dict(xy=tuple(np.asarray(p, float)), corridors={name}))
        else:
            hit["corridors"].add(name)

tss = []; new_buses = {}; next_bus_id = 58
for td in _targets:
    p = np.asarray(td["xy"], float)
    meta = dict(corridors=tuple(sorted(td["corridors"])), target_xy=tuple(p))
    b, d = nearest_bus(p)
    if d <= SNAP_KM:
        tss.append({"type": "bus", "bus": b, "xy": tuple(_P(b)),
                    "tie_km": round(d, 1), **meta})
        continue
    f, t, tt, dd, q = nearest_branch_tap(p)
    if tt < 0.05:
        tss.append({"type": "bus", "bus": f, "xy": tuple(_P(f)),
                    "tie_km": round(np.linalg.norm(p - _P(f)) * U2KM, 1), **meta})
    elif tt > 0.95:
        tss.append({"type": "bus", "bus": t, "xy": tuple(_P(t)),
                    "tie_km": round(np.linalg.norm(p - _P(t)) * U2KM, 1), **meta})
    else:
        nb = next_bus_id; next_bus_id += 1; new_buses[nb] = tuple(q)
        tss.append({"type": "tap", "bus": nb, "xy": tuple(q), "split": (f, t),
                    "frac": round(tt, 3), "tie_km": round(dd, 1), **meta})

# Dedupe TSSs that resolve to the same electrical bus while preserving all corridor memberships.
_keep = []; _at = {}
for s in tss:
    k = (s["type"], s["bus"])
    if k in _at:
        q = _keep[_at[k]]
        q["corridors"] = tuple(sorted(set(q["corridors"]) | set(s["corridors"])))
        continue
    _at[k] = len(_keep); _keep.append(s)
tss = _keep
for i, s in enumerate(tss):
    s["id"] = f"TSS{i+1:02d}"

TSS_BY_CORRIDOR = {
    name: [i for i, s in enumerate(tss) if name in set(s.get("corridors", ())) ]
    for name in corridors
}
for name, ids in TSS_BY_CORRIDOR.items():
    if not ids:
        raise RuntimeError(f"No physical TSS retained for corridor {name!r}")

n_bus = sum(1 for s in tss if s["type"] == "bus")
n_tap = sum(1 for s in tss if s["type"] == "tap")
print(f"TSS placed: {len(tss)}  |  {n_bus} at existing IEEE-57 buses  |  {n_tap} new tapped buses (ids {sorted(new_buses)})")
print("Tapped lines (from,to @ fraction):", [(s['split'], s['frac']) for s in tss if s['type'] == 'tap'])
print("Corridor-restricted TSS memberships:", {k: len(v) for k, v in TSS_BY_CORRIDOR.items()})

In [ ]:
# ---- physical TSS spacing along each corridor ----
def _project_km_to_corridor(cb, p):
    """Longitudinal km coordinate of the nearest projection of point p onto corridor cb."""
    c = corridor_cumkm(cb); p = np.asarray(p, float)
    best_d2, best_km = np.inf, 0.0
    for i in range(len(cb)-1):
        A, B = _P(cb[i]), _P(cb[i+1]); AB = B-A
        den = float(np.dot(AB, AB))
        u = 0.0 if den <= 0 else float(np.clip(np.dot(p-A, AB)/den, 0.0, 1.0))
        q = A + u*AB; d2 = float(np.dot(p-q, p-q))
        if d2 < best_d2:
            best_d2 = d2; best_km = float(c[i] + u*(c[i+1]-c[i]))
    return best_km

TSS_CHAIN_KM, TSS_GAPS_KM = {}, {}
print(f"{'corridor':<30}{'len(km)':>8}{'physical TSS':>14}{'mean gap':>11}{'gap range':>17}")
print('-'*80)
for name, cb in corridors.items():
    L = float(corridor_cumkm(cb)[-1])
    kms = sorted({_project_km_to_corridor(cb, tss[j]['xy']) for j in TSS_BY_CORRIDOR[name]})
    gaps = np.diff(kms) if len(kms) > 1 else np.array([], float)
    TSS_CHAIN_KM[name] = np.asarray(kms, float); TSS_GAPS_KM[name] = np.asarray(gaps, float)
    mg = float(gaps.mean()) if len(gaps) else np.nan
    rg = f"{gaps.min():.0f}-{gaps.max():.0f} km" if len(gaps) else 'n/a'
    print(f"{name:<30}{L:>8.0f}{len(kms):>14}{mg:>9.1f} km{rg:>17}")
_all_tss_gaps = np.concatenate([g for g in TSS_GAPS_KM.values() if len(g)])
TSS_GAP_MEAN_KM = float(_all_tss_gaps.mean())

ties = [s['tie_km'] for s in tss if s['type'] == 'bus']
print(f"\nBus-TSS grid ties (km): median {np.median(ties):.1f}, max {max(ties):.1f}  (<= SNAP_KM={SNAP_KM:.0f})")
print(f"New tapped buses: {len(new_buses)} -> ids {sorted(new_buses)} "
      f"(each splits one line into two, impedance prorated by 'frac')")

In [ ]:
# ---- MAP: grid + corridors + stations + TSS ----
grid_segs = [(pos57[int(f)], pos57[int(t)]) for f, t in branch0[:, 0:2].astype(int)]
fig, ax = plt.subplots(figsize=(12, 11))
ax.add_collection(LineCollection(grid_segs, linewidths=1.2, colors="0.80", zorder=1))
for (name, cb), c in zip(corridors.items(), corridor_colors.values()):
    pts = np.array([pos57[b] for b in cb])
    ax.plot(pts[:, 0], pts[:, 1], c=c, lw=2.2, alpha=0.55, zorder=2)
gx = np.array([pos57[n] for n in grid_buses])
ax.scatter(gx[:, 0], gx[:, 1], s=12, c="0.35", zorder=3)

S = np.array(list(stations.values()))
ax.scatter(S[:, 0], S[:, 1], s=55, marker="o", facecolors="white", edgecolors="k",
           linewidths=1.4, zorder=6, label=f"Stations ({len(stations)})")
tb = np.array([s["xy"] for s in tss if s["type"] == "bus"])
ax.scatter(tb[:, 0], tb[:, 1], s=95, marker="s", c="#111", zorder=7,
           label=f"TSS @ existing bus ({n_bus})")
if n_tap:
    tt = np.array([s["xy"] for s in tss if s["type"] == "tap"])
    ax.scatter(tt[:, 0], tt[:, 1], s=140, marker="^", c="#e6194B", edgecolors="k",
               linewidths=0.6, zorder=8, label=f"TSS @ new tapped bus ({n_tap})")
    for s in tss:
        if s["type"] == "tap":
            f, t = s["split"]; q = s["xy"]
            ax.plot([pos57[f][0], q[0]], [pos57[f][1], q[1]], c="#e6194B", lw=1.0, ls=":", zorder=5)
            ax.plot([q[0], pos57[t][0]], [q[1], pos57[t][1]], c="#e6194B", lw=1.0, ls=":", zorder=5)

# 100 km scale bar
bar_u = 100.0 / U2KM
x0, y0 = min(p[0] for p in pos57.values()), min(p[1] for p in pos57.values()) - 0.04
ax.plot([x0, x0 + bar_u], [y0, y0], c="k", lw=3, zorder=12)
ax.text(x0 + bar_u / 2, y0 - 0.02, "100 km", ha="center", va="top", fontsize=9)

ax.set_aspect("equal"); ax.axis("off")
ax.legend(loc="upper left", fontsize=9)
ax.set_title(f"Stations + traction substations  (span 400 km; TSS ~{TSS_SPACING_KM:.0f} km, "
             f"stations ~{STATION_SPACING_KM:.0f} km)")
plt.tight_layout(); plt.show()

## 6. Apply the taps: split impedances and emit the expanded case

Each tapped line is replaced by two (or more) prorated halves. With the tap at fraction
alpha from the *from* bus, the series impedance and line charging split alpha : (1-alpha):
`R -> alpha*R, (1-alpha)*R`, likewise `X` and `B`; totals are preserved. Thermal rating
`rateA` is **not** split (ampacity is a conductor property, identical along the line).
The new bus is a zero-injection PQ node (the TSS traction load attaches later). No physical
length or base kV is needed - only the fraction and the canonical per-unit parameters.

In [ ]:
# Tapped lines are split proportionally to length: series impedance and shunt charging
# divide alpha:(1-alpha); two cascaded pi-sections approximate the original line (Section II-A).

# group taps by the line they split (robust to >1 tap per line)
taps_by_line = defaultdict(list); newbus_line = {}; newbus_xy = {}
for s in tss:
    if s["type"] == "tap":
        taps_by_line[s["split"]].append((s["frac"], s["bus"]))
        newbus_line[s["bus"]] = s["split"]; newbus_xy[s["bus"]] = s["xy"]

# --- new bus rows: PQ, zero injection, copy area/zone/limits from the 'from' endpoint ---
new_bus_rows = []
for nb in sorted(newbus_xy):
    f, t = newbus_line[nb]
    row = bus0[bus0[:, 0].astype(int) == f][0].copy()
    row[0] = nb; row[1] = 1                       # id, PQ type
    row[2] = 0.0; row[3] = 0.0                    # Pd, Qd
    row[4] = 0.0; row[5] = 0.0                    # Gs, Bs
    row[7] = 1.0; row[8] = 0.0                    # Vm, Va
    new_bus_rows.append(row)
bus_split = np.vstack([bus0, np.array(new_bus_rows)]) if new_bus_rows else bus0.copy()

# --- branch rows: drop tapped originals, add prorated segments in order along the line ---
keep_mask = np.array([(int(r[0]), int(r[1])) not in taps_by_line for r in branch0])
seg_rows = []; split_report = []
for (f, t), lst in taps_by_line.items():
    orig = branch0[(branch0[:, 0].astype(int) == f) & (branch0[:, 1].astype(int) == t)][0]
    R, X, B = orig[2], orig[3], orig[4]
    lst = sorted(lst)                              # by fraction from f
    nodes = [f] + [nb for _, nb in lst] + [t]
    frac  = [0.0] + [a for a, _ in lst] + [1.0]
    these = []; chain = []
    for i in range(len(nodes) - 1):
        w = frac[i+1] - frac[i]
        seg = orig.copy()
        seg[0] = nodes[i]; seg[1] = nodes[i+1]
        seg[2] = w * R; seg[3] = w * X; seg[4] = w * B   # prorate R, X, B
        # A split transformer branch would keep its off-nominal ratio on the first segment.
        # None of the six tapped branches is a transformer, so every segment has ratio 0 (line).
        seg[8] = (orig[8] if (i == 0 and orig[8] not in (0.0, 1.0)) else 0.0)
        seg[9] = 0.0; seg[10] = 1.0                      # angle, status (rateA/B/C kept)
        these.append(seg); chain.append((nodes[i], nodes[i+1], round(w, 3)))
    # conservation (exact)
    assert abs(sum(r[2] for r in these) - R) < 1e-9
    assert abs(sum(r[3] for r in these) - X) < 1e-9
    assert abs(sum(r[4] for r in these) - B) < 1e-9
    seg_rows += these; split_report.append((f, t, chain))

branch_split = np.vstack([branch0[keep_mask], np.array(seg_rows)])

mpc_split = {"version": "2", "baseMVA": baseMVA, "bus": bus_split,
             "gen": gen0_full, "branch": branch_split, "gencost": gencost0_full}

print(f"Expanded case: buses {bus0.shape[0]} -> {bus_split.shape[0]}, "
      f"branches {branch0.shape[0]} -> {branch_split.shape[0]}  "
      f"({len(taps_by_line)} lines -> {len(seg_rows)} segments)")
print("Series R/X and charging B conserved on every tapped line (asserted).")
for f, t, chain in split_report:
    print(f"  line {f}-{t}:  " + "  ".join(f"{a}-{b} (x{w})" for a, b, w in chain))

In [ ]:
# --- validation: power flow on original vs expanded case ---
from pypower.api import case57, runpf, ppoption
opt = ppoption(VERBOSE=0, OUT_ALL=0)
c_orig = case57(); c_orig["branch"][:, 5] = pglib_rateA
r0, ok0 = runpf(c_orig, opt)
r1, ok1 = runpf(mpc_split, opt)
orig_ids = [int(b[0]) for b in bus0]
v0 = {int(b[0]): b[7] for b in r0["bus"]}; v1 = {int(b[0]): b[7] for b in r1["bus"]}
a0 = {int(b[0]): b[8] for b in r0["bus"]}; a1 = {int(b[0]): b[8] for b in r1["bus"]}
dV = max(abs(v0[i] - v1[i]) for i in orig_ids)
dA = max(abs(a0[i] - a1[i]) for i in orig_ids)
print(f"PF converged  (original={bool(ok0)}, expanded={bool(ok1)})")
print(f"max |dVm| at original buses: {dV:.2e} pu   |   max |dVa|: {dA:.2e} deg")
print("-> the expanded case reproduces the original solution. The small residual comes from representing\n"
      "   each tapped line by two cascaded pi sections instead of one.")

In [ ]:
# --- emit a ready-to-use MATPOWER case + CSVs ---
import os
NL = chr(10)
def _fmt(a):
    return (";" + NL).join("  " + " ".join(f"{v:.6g}" for v in row) for row in a)
def write_matpower_m(path, mpc, name):
    blocks = [f"function mpc = {name}", "mpc.version = '2';",
              f"mpc.baseMVA = {mpc['baseMVA']:.6g};",
              "mpc.bus = [",     _fmt(mpc["bus"]),    "];",
              "mpc.gen = [",     _fmt(mpc["gen"]),    "];",
              "mpc.branch = [",  _fmt(mpc["branch"]), "];",
              "mpc.gencost = [", _fmt(mpc["gencost"]),"];"]
    with open(path, "w") as fh:
        fh.write(NL.join(blocks) + NL)

write_matpower_m("case57_hsr.m", mpc_split, "case57_hsr")
np.savetxt("case57_hsr_bus.csv",    mpc_split["bus"],    delimiter=",")
np.savetxt("case57_hsr_branch.csv", mpc_split["branch"], delimiter=",")
print("Emitted (in", os.getcwd() + "):")
print("  case57_hsr.m           - MATPOWER case, 63 buses / 86 branches")
print("  case57_hsr_bus.csv     - bus matrix")
print("  case57_hsr_branch.csv  - branch matrix")

## 7. Train-movement simulation and catenary voltage check

A full 24 h simulation at **1 s** resolution: end-to-end services both directions on all six
corridors, stopping at every station, on a peak/off-peak schedule.

**Trainset** (CR400AF-like 8-car), calibrated so 0->350 km/h takes ~391 s (the published
Fuxing figure [23]) and cruise energy is ~21 kWh/km. Longitudinal dynamics use the **Davis
resistance** `R(v)=A+Bv+Cv^2`, an adhesion/constant-power tractive-effort curve, and
service braking with regeneration capped to the line.

**Demand vs regen are kept strictly separate** (never a single net trace): at each 1 s step
we split each section's power into motoring `m` and braking `b`, do same-instant in-section
reuse `r=min(sum m, sum b)`, and carry `D=sum m - r` (demand to supply) and
`R=sum b - r` (surplus regen -> storage/export/spill) as two series. Reuse is bounded by
the **neutral-section arm** (~24 km), across which trains cannot share power.

**Catenary voltage check.** A radial voltage-drop estimate on each section, with the equivalent 2x25 kV monovoltage
impedance `z = 0.04 + j0.10 ohm/km`, confirms that the catenary stays within the 19-27.5 kV band.

In [ ]:
# ===== Calibrated CR400AF "Fuxing"-like 8-car EMU ============================
# Values as declared in Section II-B of the paper.
MASS=473e3                 # kg, loaded mass of the reference trainset [23]
M_EFF=MASS*1.08            # +8% rotational-inertia allowance (declared assumption)
P_WHEEL=9.8e6              # W wheel traction power, calibrated to the 391 s 0->350 km/h figure [23]
F_START=340e3              # N starting tractive effort (adhesion/force limited)
V_MAX_T=350/3.6            # m/s = 350 km/h maximum operating speed [23]
A_BRAKE=0.5                # m/s^2 service deceleration (declared assumption)
P_REGEN_MAX=9.6e6          # W regeneration cap at the traction-converter rating; the rest goes to friction braking
ETA_MOT=0.88; ETA_REG=0.85 # pantograph<->wheel drivetrain efficiencies (declared assumption)
P_AUX=300e3                # W auxiliary/hotel load (HVAC, lighting, passenger services)
# Davis running resistance R(v)=A+Bv+Cv^2 [N], v in m/s [24]. Coefficients calibrated to reproduce
# the published 391 s 0->350 km/h acceleration and ~21 kWh/km cruise energy of the reference trainset [23].
A_D,B_D,C_D=5.5e3,70.0,5.6
def davis(v): return A_D+B_D*v+C_D*v*v

# ===== 2x25 kV AT traction network (monovoltage-equivalent) =================
# The 2x25 kV autotransformer system is represented by one equivalent series impedance per km.
# 0.04+j0.10 ohm/km is the representative 50 Hz value declared in Section II-B of the paper.
Z_EQ=complex(0.04,0.10)
PF=0.98                          # fixed traction power factor (declared convention, Section II-A)
ZDROP=Z_EQ.real+np.tan(np.arccos(PF))*Z_EQ.imag   # for |dV| ~ (P*R+Q*X)/V, ohm/km
V_NOM=25e3; V_MAX_CAT=27.5e3; V_MIN_CAT=19e3       # EN 50163 Un, Umax1, Umin1 [22]
SEC_LEN_KM=24.0                  # neutral-section arm length [1] (value is a declared assumption)
DWELL=120.0                      # s intermediate-station scheduled dwell (declared assumption)


def drive_cycle(stations_km, dt=1.0):
    pos=[]; m=[]; b=[]
    for i in range(len(stations_km)-1):
        x=stations_km[i]*1000.0; target=stations_km[i+1]*1000.0; v=0.0
        while x < target-0.5:
            dist=target-x; bdist=v*v/(2*A_BRAKE)
            if dist<=bdist:
                Fb=max(M_EFF*A_BRAKE-davis(v),0.0)
                bb=min(Fb*v*ETA_REG,P_REGEN_MAX); mm=P_AUX; a=-A_BRAKE
            elif v<V_MAX_T:
                F=F_START if v<=0 else min(F_START,P_WHEEL/v)
                a=(F-davis(v))/M_EFF; mm=(F*v)/ETA_MOT+P_AUX; bb=0.0
            else:
                F=davis(v); a=0.0; mm=(F*v)/ETA_MOT+P_AUX; bb=0.0
            v=max(v+a*dt,0.0); x+=v*dt
            pos.append(x/1000.0); m.append(mm/1e6); b.append(bb/1e6)
        if i < len(stations_km)-2:
            pos += [stations_km[i+1]]*int(DWELL)
            m   += [P_AUX/1e6]*int(DWELL)
            b   += [0.0]*int(DWELL)
    return np.array(pos), np.array(m), np.array(b)


# Operational headways 6/10/15 min (peak/shoulder/off-peak) over a 06:00-24:00 window, and a
# 180 s minimum separation between successive trains (declared assumptions).
MIN_HEADWAY_S = 180
# Twelve corridor-direction streams are phased at 30 s increments rather than all departing on
# the same clock second. This keeps the same service frequency while removing artificial
# system-wide synchronization from the nominal timetable.
STREAM_PHASE_STEP_S = 30
# Include the preceding service day when constructing a 24 h trace, then retain only the target
# day. This carries late trains through midnight without double-counting the next day.
CYCLIC_DAY_OFFSETS = (-1, 0)


def headway_at(s):
    h=s/3600.0
    if 6<=h<9 or 17<=h<20: return 360
    if 9<=h<17: return 600
    if 20<=h<24: return 900
    return None


def stream_phase_s(corridor_index, direction):
    """Fixed nominal phase for one corridor/direction stream, in seconds."""
    stream = 2*int(corridor_index) + int(direction)
    return int((stream*STREAM_PHASE_STEP_S) % 360)


def departures(phase_s=0, day_offset=0):
    """Scheduled origin departures for one corridor-direction stream."""
    ts=[]; t=6*3600
    while t<24*3600:
        hw=headway_at(t)
        if hw is None: break
        ts.append(int(t + phase_s + day_offset*86400))
        t += hw
    return ts


# self-check against the 391 s benchmark
def _accel_time_s():
    v=t=0.0
    while v<V_MAX_T:
        F=F_START if v<=0 else min(F_START,P_WHEEL/v)
        a=(F-davis(v))/M_EFF
        if a<=0: break
        v+=a*0.1; t+=0.1
    return t
_t=_accel_time_s(); assert 370<_t<410, _t
assert MIN_HEADWAY_S <= min(x for x in (360, 600, 900))
print(f"trainset calibration: 0->350 km/h in {_t:.0f} s (Fuxing ~391 s)  OK")
print(f"nominal timetable: 12 phased streams, {MIN_HEADWAY_S}s minimum operational separation, "
      "cyclic carry-over across midnight")

In [ ]:
# ---- build per-corridor structure, canonical section-to-TSS map, and cyclic nominal day ----
def corr_len(cb):
    c=0.0
    for i in range(len(cb)-1):
        c+=np.hypot(pos57[cb[i+1]][0]-pos57[cb[i]][0],
                    pos57[cb[i+1]][1]-pos57[cb[i]][1])*U2KM
    return c

N=86400; deps=departures()   # unphased reference stream used only for daily run counts
corr=[]; nsec_tot=0
for name,cb in corridors.items():
    L=corr_len(cb); nsec=max(1,round(L/SEC_LEN_KM))
    nst=max(2,round(L/STATION_SPACING_KM)); stations=list(np.linspace(0,L,nst+1))
    ntss_sites=len(TSS_BY_CORRIDOR[name])
    corr.append(dict(name=name,L=L,nsec=nsec,seclen=L/nsec,ntss=ntss_sites,
                     stations=stations,soff=nsec_tot))
    nsec_tot+=nsec

# Canonical electrical ordering: one row per physical TSS in the `tss` list. Every section is
# assigned only among TSSs tagged to its own corridor. Assignment uses longitudinal distance along
# that corridor rather than global XY distance, so crossings cannot transfer a section to another
# corridor. This single map is reused by nominal D/R, jitter days, planning envelopes, figures, and
# every validation layer.
coupling = [(int(s["bus"]), np.asarray(s["xy"], float)) for s in tss]
cb_ids   = [b for b,_ in coupling]
cb_xy    = np.asarray([xy for _,xy in coupling], float)
SECTION_TO_TSS = np.full(nsec_tot, -1, dtype=int)
for cd in corr:
    eligible = np.asarray(TSS_BY_CORRIDOR[cd["name"]], dtype=int)
    cb = corridors[cd["name"]]
    eligible_km = np.asarray([_project_km_to_corridor(cb, tss[int(j)]["xy"]) for j in eligible], float)
    for sec_idx in range(cd["nsec"]):
        midpoint_km = (sec_idx + 0.5) * cd["seclen"]
        local = int(np.argmin(np.abs(eligible_km - midpoint_km)))
        SECTION_TO_TSS[cd["soff"] + sec_idx] = int(eligible[local])
assert (SECTION_TO_TSS >= 0).all()
for cd in corr:
    _rows = SECTION_TO_TSS[cd["soff"]:cd["soff"]+cd["nsec"]]
    assert set(map(int, _rows)).issubset(set(TSS_BY_CORRIDOR[cd["name"]]))
SECTION_TO_BUS = np.asarray([cb_ids[j] for j in SECTION_TO_TSS], dtype=int)
print(f"canonical section-to-TSS allocation: {nsec_tot} sections -> {len(coupling)} physical TSSs; "
      "cross-corridor assignment is prohibited")
print("  TSSs actually supplying sections by corridor:",
      {cd["name"]: len(set(map(int, SECTION_TO_TSS[cd["soff"]:cd["soff"]+cd["nsec"]]))) for cd in corr})

M=np.zeros((nsec_tot,N),np.float32); B=np.zeros((nsec_tot,N),np.float32)
_nominal_carry=0; _nominal_runs=0
for _ci, cd in enumerate(corr):
    p0,m0,b0=drive_cycle(cd["stations"]); Lp=len(p0)
    for direction in (0,1):
        p = p0 if direction==0 else (cd["L"]-p0)
        sec = cd["soff"]+np.clip((p/cd["seclen"]).astype(int),0,cd["nsec"]-1)
        phase = stream_phase_s(_ci, direction)
        _nominal_runs += len(departures(phase, 0))
        for _day in CYCLIC_DAY_OFFSETS:
            for t0 in departures(phase, _day):
                tt=t0+np.arange(Lp); ok=(tt>=0)&(tt<N)
                if _day < 0 and ok.any(): _nominal_carry += 1
                if ok.any():
                    np.add.at(M,(sec[ok],tt[ok]),m0[ok])
                    np.add.at(B,(sec[ok],tt[ok]),b0[ok])

# same-instant in-section reuse, then separate demand and surplus regen (never a
# single net trace; reuse bounded by the neutral-section arm) [4]
reuse=np.minimum(M,B); D=M-reuse; R=B-reuse
assert _nominal_runs == len(corr)*2*len(deps)
print(f"accumulated {_nominal_runs} current-day train-runs plus {_nominal_carry} prior-day "
      f"carry-over runs into {nsec_tot} sections x {N} s")
print("stream phase offsets [s]:", [stream_phase_s(i,d) for i in range(len(corr)) for d in (0,1)])

In [ ]:
# ---- aggregate to the canonical physical-TSS ordering, catenary voltages, day statistics ----
GRID_LOAD_MW=float(bus0[:,2].sum())   # IEEE-57 total active load (~1251 MW) [21]
Dt=np.zeros((len(coupling),N),np.float32); Rt=np.zeros((len(coupling),N),np.float32)
for g in range(nsec_tot):
    j=int(SECTION_TO_TSS[g])
    Dt[j]+=D[g]; Rt[j]+=R[g]

# Per-section radial catenary voltage: |dV| ~ (P*R + Q*X)/V, with Q=P*tan(phi) folded
# into ZDROP (fixed 0.98 power factor); load taken at effective distance seclen/2 from TSS.
vmin=V_NOM; vmax=V_NOM
for cd in corr:
    deff=cd["seclen"]/2.0
    for s in range(cd["nsec"]):
        g=cd["soff"]+s
        vmin=min(vmin,(V_NOM-(D[g]*1e6/V_NOM)*ZDROP*deff).min())
        vmax=max(vmax,(V_NOM+(R[g]*1e6/V_NOM)*ZDROP*deff).max())

sysD=D.sum(0); sysR=R.sum(0); eD=sysD.sum()/3600; eR=R.sum()/3600; eRe=reuse.sum()/3600
gross=eD+eRe
assert np.allclose(Dt.sum(0), sysD, atol=1e-4)
assert np.allclose(Rt.sum(0), sysR, atol=1e-4)
print(f"Scheduled origin departures/day {_nominal_runs} across 12 streams "
      f"(+{_nominal_carry} preceding-day trains contributing after midnight) | "
      f"demand {eD:,.0f} MWh, surplus regen {eR:,.0f} MWh, in-section reuse {eRe:,.0f} MWh")
print(f"Regen: {eRe/gross*100:.0f}% reused same-instant, {eR/gross*100:.0f}% surplus "
      f"(-> storage/export/spill decision)")
print(f"System peak demand {sysD.max():.0f} MW ({sysD.max()/GRID_LOAD_MW*100:.0f}% of grid) at "
      f"{sysD.argmax()/3600:.1f} h | peak surplus regen {sysR.max():.0f} MW")
nominal_roll10 = np.roll(sysD, -10) - sysD
print(f"Max rolling 10 s demand excursion {np.max(np.abs(nominal_roll10)):.0f} MW | "
      f"per-TSS peak demand max {Dt.max(1).max():.0f} MW")
print(f"Catenary voltage {vmin/1e3:.1f}-{vmax/1e3:.1f} kV (limits {V_MIN_CAT/1e3:.0f}-{V_MAX_CAT/1e3:.1f})")
print(f"Coupling stress regime: rail energy is {eD/(GRID_LOAD_MW*24)*100:.1f}% of base-grid daily energy; "
      f"rail peak is {sysD.max()/GRID_LOAD_MW*100:.1f}% of base-grid peak load (deliberately strong coupling).")

# Export exactly the per-TSS signature used by planning and validation.
np.savez("hsr_demand_signature.npz", Dt=Dt, Rt=Rt, dt_s=1,
         tss_bus=np.asarray(cb_ids,int), section_to_tss=SECTION_TO_TSS,
         section_to_bus=SECTION_TO_BUS)
print("saved hsr_demand_signature.npz (canonical physical-TSS demand/regen signature, 1 s)")

In [ ]:
# ---- plots: full-day system profile + a 20-min zoom on the busiest TSS ----
hrs=np.arange(N)/3600.0
fig,ax=plt.subplots(2,1,figsize=(13,8))
ax[0].plot(hrs,sysD,lw=0.6,c="#c1121f",label="Demand D")
ax[0].plot(hrs,-sysR,lw=0.6,c="#2a9d8f",label="Surplus regen R (negative)")
ax[0].set_xlim(0,24); ax[0].set_xlabel("hour of day"); ax[0].set_ylabel("MW")
ax[0].set_title("System-wide traction demand & surplus regen (full day, 1 s resolution)")
ax[0].legend(loc="upper right",fontsize=8); ax[0].grid(alpha=0.3)
kpk=Dt.max(1).argmax(); c0=max(0,Dt[kpk].argmax()-600); win=slice(c0,c0+1200)
ax[1].plot(np.arange(1200)/60,Dt[kpk][win],c="#c1121f",lw=0.9,label="Demand")
ax[1].plot(np.arange(1200)/60,-Rt[kpk][win],c="#2a9d8f",lw=0.9,label="Surplus regen")
ax[1].set_xlabel("minutes into window"); ax[1].set_ylabel("MW")
ax[1].set_title(f"Busiest TSS (#{kpk}): 20-min zoom - the 1 s spikes net-averaging would erase")
ax[1].legend(loc="upper right",fontsize=8); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 8. Results, figures, and verification

Tables and figures built from the Section 7 simulation. Figures are saved as PNG and PDF. Table 2 places the main
modelling choices against published values [22], [23] and against the design targets of the benchmark.


In [ ]:
# ---- Tables 1-3 ----
def _detailed_run(stn, dt=1.0):
    T=[];Vv=[];Pe=[];tc=0.0
    for i in range(len(stn)-1):
        x=stn[i]*1000.0; tgt=stn[i+1]*1000.0; v=0.0
        while x<tgt-0.5:
            dist=tgt-x
            if dist<=v*v/(2*A_BRAKE):
                Fb=max(M_EFF*A_BRAKE-davis(v),0.0); bb=min(Fb*v*ETA_REG,P_REGEN_MAX)
                pe=-(bb-P_AUX); a=-A_BRAKE
            elif v<V_MAX_T:
                F=F_START if v<=0 else min(F_START,P_WHEEL/v); a=(F-davis(v))/M_EFF
                pe=(F*v)/ETA_MOT+P_AUX
            else:
                F=davis(v); a=0.0; pe=(F*v)/ETA_MOT+P_AUX
            v=max(v+a*dt,0.0); x+=v*dt; tc+=dt; T.append(tc);Vv.append(v*3.6);Pe.append(pe/1e6)
        if i<len(stn)-2:
            for _ in range(int(DWELL)): tc+=1;T.append(tc);Vv.append(0.0);Pe.append(P_AUX/1e6)
    return np.array(T),np.array(Vv),np.array(Pe)

corr_stats=[]
for cd in corr:
    p0,m0,b0=drive_cycle(cd["stations"]); L=cd["L"]
    corr_stats.append((cd["name"],L,len(cd["stations"]),cd["nsec"],cd["ntss"],
                       len(p0)/60.0,L/(len(p0)/3600.0),(m0.sum()/3600)*1000/L,b0.sum()/m0.sum()*100))
print("Table 1  Per-corridor summary")
print(f"{'corridor':<30}{'km':>5}{'stn':>4}{'sec':>4}{'TSS':>4}{'min':>5}{'km/h':>6}{'kWh/km':>8}{'regen%':>7}")
for s in corr_stats:
    print(f"{s[0]:<30}{s[1]:>5.0f}{s[2]:>4}{s[3]:>4}{s[4]:>4}{s[5]:>5.0f}{s[6]:>6.0f}{s[7]:>8.1f}{s[8]:>7.0f}")

vmn=25.0
for cd in corr:
    dfe=cd["seclen"]/2
    for s in range(cd["nsec"]):
        vmn=min(vmn,(V_NOM-(D[cd["soff"]+s]*1e6/V_NOM)*ZDROP*dfe).min()/1e3)
ramp=np.abs(np.roll(sysD, -10)-sysD)
ver=[("0->350 km/h accel time",f"{_accel_time_s():.0f} s","391 s","[23]"),
     ("cruise energy @350",f"~{(davis(V_MAX_T)*V_MAX_T/ETA_MOT+P_AUX)/1000/(V_MAX_T*3.6):.0f} kWh/km","~21 kWh/km","[23]"),
     ("regen recovery / gross draw",f"{(eRe+eR)/gross*100:.0f}%","reported only","model output"),
     ("all-stop journey speed",f"{np.mean([s[6] for s in corr_stats]):.0f} km/h","200-250 km/h","design target"),
     ("catenary voltage (min)",f"{vmn:.1f} kV",">=19 kV","[22]"),
     ("impedance-split PF error",f"{dV:.2e} pu","~0 (transparent)","Section 6"),
     ("TSS spacing achieved",f"{TSS_GAP_MEAN_KM:.0f} km","~50 km","design target"),
     ("peak traction load",f"{sysD.max():.0f} MW ({sysD.max()/GRID_LOAD_MW*100:.0f}%)","<100% of grid","design")]
print("\nTable 2  Verification against published values and design targets")
print(f"{'quantity':<34}{'this model':<17}{'benchmark':<19}{'src'}")
for q,m,b,s in ver: print(f"{q:<34}{m:<17}{b:<19}{s}")

res=[("Scheduled origin departures / day",f"{_nominal_runs}"),
     ("Preceding-day trains active after midnight",f"{_nominal_carry}"),
     ("Traction demand energy",f"{eD:,.0f} MWh"),
     ("Same-instant reuse",f"{eRe:,.0f} MWh ({eRe/gross*100:.0f}%)"),
     ("Surplus regen",f"{eR:,.0f} MWh ({eR/gross*100:.0f}%)"),
     ("System peak demand",f"{sysD.max():.0f} MW ({sysD.max()/GRID_LOAD_MW*100:.0f}% of grid)"),
     ("System peak surplus regen",f"{sysR.max():.0f} MW"),
     ("Max rolling 10 s demand excursion",f"{ramp.max():.0f} MW"),
     ("Per-TSS peak demand (max)",f"{Dt.max(1).max():.0f} MW"),
     ("Catenary voltage window",f"{vmn:.1f}-{vmax/1e3:.1f} kV")]
print("\nTable 3  Simulation results summary")
for k,v in res: print(f"  {k:<44}{v}")

In [ ]:
# ---- Figure 1: physics verification ----
fig,ax=plt.subplots(2,2,figsize=(11,7))
T,Vv,Pe=_detailed_run([0,45,90])
ax[0,0].plot(T,Vv,c="#1f77b4"); ax[0,0].axhline(350,ls=":",c="0.5")
ax[0,0].set(xlabel="time (s)",ylabel="speed (km/h)",title="(a) Speed, 45 km inter-station legs")
ax[0,1].plot(T,Pe,c="#c1121f"); ax[0,1].axhline(0,c="0.5",lw=0.8)
ax[0,1].set(xlabel="time (s)",ylabel="pantograph power (MW)",title="(b) Power (+draw / -regen)")
vv=np.linspace(1,V_MAX_T,300)
ax[1,0].plot(vv*3.6,[min(F_START,P_WHEEL/v)/1e3 for v in vv],c="#1f77b4",label="tractive effort")
ax[1,0].plot(vv*3.6,[davis(v)/1e3 for v in vv],c="#c1121f",label="Davis resistance")
ax[1,0].set(xlabel="speed (km/h)",ylabel="force (kN)",title="(c) Traction & resistance vs speed"); ax[1,0].legend(fontsize=8)
ek=[s[7] for s in corr_stats]
ax[1,1].bar(range(len(corr_stats)),ek,color=["#1f77b4","#ff7f0e","#2ca02c","#d62728","#9467bd","#8c564b"])
ax[1,1].axhline(21,color="0.6",ls="--",zorder=0)
ax[1,1].set_xticks(range(len(corr_stats))); ax[1,1].set_xticklabels([s[0].split()[0] for s in corr_stats])
ax[1,1].set(ylabel="kWh/km",title="(d) All-stop journey energy (dashed = ~21 kWh/km cruise [23])")
ax[1,1].text(0.02,0.96,"Acceleration, dwell auxiliaries and braking make all-stop values higher",
             transform=ax[1,1].transAxes,va="top",fontsize=8)
plt.tight_layout(); plt.savefig("fig1_physics.pdf"); plt.savefig("fig1_physics.png"); plt.show()

In [ ]:
# ---- Figure 2: system signature, load-duration, spatial ----
fig=plt.figure(figsize=(15,4.4)); hrs=np.arange(N)/3600.0
a0=fig.add_subplot(1,3,1)
a0.plot(hrs,sysD,lw=0.5,c="#c1121f",label="demand D"); a0.plot(hrs,-sysR,lw=0.5,c="#2a9d8f",label="surplus regen R")
a0.set(xlim=(0,24),xlabel="hour",ylabel="MW",title="(a) System demand & surplus regen"); a0.legend(fontsize=8); a0.grid(alpha=0.3)
a1=fig.add_subplot(1,3,2)
ld=np.sort(sysD)[::-1]
a1.plot(np.arange(N)/N*100,ld,c="#c1121f"); a1.axhline(GRID_LOAD_MW,ls="--",c="0.4",label=f"grid {GRID_LOAD_MW:.0f} MW")
a1.set(xlabel="% of day exceeded",ylabel="MW",title="(b) Traction load-duration curve"); a1.legend(fontsize=8); a1.grid(alpha=0.3)
a2=fig.add_subplot(1,3,3)
tss_pos=np.asarray(cb_xy, float)   # same physical-TSS ordering as Dt and Dbus
for cb,cl in zip(corridors.values(),["#1f77b4","#ff7f0e","#2ca02c","#d62728","#9467bd","#8c564b"]):
    p=np.array([pos57[b] for b in cb]); a2.plot(p[:,0],p[:,1],c=cl,lw=1.4,alpha=0.5)
gx=np.array([pos57[n] for n in grid_buses]); a2.scatter(gx[:,0],gx[:,1],s=6,c="0.6")
pk=Dt.max(1); sc=a2.scatter(tss_pos[:,0],tss_pos[:,1],s=20+pk*6,c=pk,cmap="plasma",edgecolors="k",linewidths=0.4)
plt.colorbar(sc,ax=a2,label="TSS peak demand (MW)",shrink=0.8)
a2.set_aspect("equal"); a2.axis("off"); a2.set_title("(c) Spatial TSS peak demand")
plt.tight_layout(); plt.savefig("fig2_signature.pdf"); plt.savefig("fig2_signature.png"); plt.show()

In [ ]:
# ---- Figure 3: dynamics & correctness (ramp, why-separate, feeder voltage) ----
fig,ax=plt.subplots(1,3,figsize=(14,4.2))
ax[0].hist(ramp,bins=60,color="#5a189a"); ax[0].set_yscale("log")
ax[0].set(xlabel="|dP| per 10 s (MW)",ylabel="count",title=f"(a) Demand ramp dist. (max {ramp.max():.0f} MW)")
kpk=Dt.max(1).argmax(); c0=max(0,Dt[kpk].argmax()-300); w=slice(c0,c0+900); tt=np.arange(900)/60
ax[1].plot(tt,Dt[kpk][w],c="#c1121f",label="demand D"); ax[1].plot(tt,-Rt[kpk][w],c="#2a9d8f",label="surplus regen R")
ax[1].plot(tt,Dt[kpk][w]-Rt[kpk][w],c="0.4",ls=":",label="naive net D-R")
ax[1].set(xlabel="minutes",ylabel="MW",title=f"(b) TSS #{kpk}: separation vs naive net"); ax[1].legend(fontsize=7); ax[1].grid(alpha=0.3)
dd=np.linspace(0,SEC_LEN_KM,50)
for Pmw,ls in [(20,"-"),(40,"--"),(60,":")]:
    ax[2].plot(dd,(V_NOM-(Pmw*1e6/V_NOM)*ZDROP*dd)/1e3,ls=ls,c="#1f77b4",label=f"{Pmw} MW")
ax[2].axhline(V_MIN_CAT/1e3,c="r",lw=0.8); ax[2].axhline(V_NOM/1e3,c="0.5",lw=0.6)
ax[2].set(xlabel="distance from TSS (km)",ylabel="catenary voltage (kV)",title="(c) Feeder voltage vs load"); ax[2].legend(fontsize=7); ax[2].grid(alpha=0.3)
plt.tight_layout(); plt.savefig("fig3_dynamics.pdf"); plt.savefig("fig3_dynamics.png"); plt.show()

## 9. Transmission AC diagnostics under the nominal train demand

The one-second traction demand is mapped to the grid buses with the same section-to-TSS allocation used by the
planning model. Surplus regeneration is excluded in this draw-direction diagnostic. Power flows are solved at the 24
hourly points and at the one-second system peak, and their results are reported only at these sampled points.
Section 9 uses a simple proportional-dispatch power flow. Section 9a asks whether an AC OPF can keep the required
voltage band at the same points.


In [ ]:
from pypower.api import runpf, ppoption

busids   = mpc_split["bus"][:,0].astype(int)
base_vec = mpc_split["bus"][:,2].copy()
Pmax     = gen0_full[:,8].copy(); SUMPMAX = float(Pmax.sum())
Vmin_lim, Vmax_lim = 0.94, 1.06
ppopt = ppoption(VERBOSE=0, OUT_ALL=0)
BASELINE_SAMPLE_STEP_S = 3600

cbrows = np.array([np.where(busids==b)[0][0] for b in cb_ids])
Dbus = np.asarray(Dt, float).copy()
assert Dbus.shape == (len(coupling), N)
assert np.allclose(Dbus.sum(0), sysD, atol=1e-4)
print(f"Coupling buses: {len(coupling)} ({sum(1 for s in tss if s['type']=='tap')} taps). "
      f"Canonical-mapped traction peak {Dbus.sum(0).max():.0f} MW.")

def solve_pf(ti):
    load = base_vec.copy(); load[cbrows] += Dbus[:, int(ti)]
    total = float(load.sum()); loading_fraction = total/SUMPMAX
    bt = mpc_split["bus"].copy(); gt = gen0_full.copy()
    bt[:,2] = load; gt[:,1] = Pmax*min(loading_fraction, 1.0)
    case_pf = {"version":"2","baseMVA":baseMVA,"bus":bt,"gen":gt,
               "branch":mpc_split["branch"].copy(),"gencost":gencost0_full}
    try:
        result, ok = runpf(case_pf, ppopt)
    except Exception:
        return False, np.nan, np.nan, np.nan, loading_fraction, np.nan, total
    if not bool(ok):
        return False, np.nan, np.nan, np.nan, loading_fraction, np.nan, total
    Vm = np.asarray(result["bus"][:,7], float)
    Sf = np.abs(result["branch"][:,13] + 1j*result["branch"][:,14])
    rateA = np.asarray(result["branch"][:,5], float)
    overloads = int(((rateA > 0) & (Sf > rateA + 1e-6)).sum())
    return True, float(Vm.min()), float(Vm.mean()), float(Vm.max()), loading_fraction, overloads, total

# No-traction reference.
base_case_pf = {"version":"2","baseMVA":baseMVA,"bus":mpc_split["bus"].copy(),
                "gen":gen0_full.copy(),"branch":mpc_split["branch"].copy(),
                "gencost":gencost0_full}
base_case_pf["bus"][:,2] = base_vec
try:
    base_result, base_ok = runpf(base_case_pf, ppopt)
    base_vmin = float(base_result["bus"][:,7].min()) if bool(base_ok) else np.nan
except Exception:
    base_vmin = np.nan

samples = sorted(set(range(0, N, BASELINE_SAMPLE_STEP_S)) | {int(sysD.argmax())})
S = [solve_pf(ti) for ti in samples]
tsh   = np.asarray(samples, float)/3600.0
conv  = np.asarray([x[0] for x in S], bool)
vmn   = np.asarray([x[1] for x in S], float)
vmean = np.asarray([x[2] for x in S], float)
vmx   = np.asarray([x[3] for x in S], float)
lf    = np.asarray([x[4] for x in S], float)
ol    = np.asarray([x[5] for x in S], float)
tot   = np.asarray([x[6] for x in S], float)

gen_deficit_h = float((sysD > (SUMPMAX-base_vec.sum())).sum())/3600.0
print(f"\nSampled PF points: {len(samples)} hourly/peak snapshots; converged {int(conv.sum())}/{len(samples)}")
print(f"No-traction sampled reference: Vmin={base_vmin:.3f} pu" if np.isfinite(base_vmin)
      else "No-traction PF reference did not converge.")
if conv.any():
    print(f"Converged traction snapshots: Vmin {np.nanmin(vmn):.3f} | mean-of-bus-means "
          f"{np.nanmean(vmean):.3f} | Vmax {np.nanmax(vmx):.3f} pu")
    print(f"Sampled points below 0.90 pu: {int(np.count_nonzero(conv & (vmn<0.90)))}/{int(conv.sum())}")
    print(f"Sampled points with a thermal overload: {int(np.count_nonzero(conv & (ol>0)))}/{int(conv.sum())}; "
          f"worst count {int(np.nanmax(ol))}")
print(f"One-second generation-capacity deficit duration: {gen_deficit_h:.2f} h/day")
print(f"Peak total load {np.nanmax(tot):.0f} MW vs installed nameplate {SUMPMAX:.0f} MW "
      f"({np.nanmax(tot)-SUMPMAX:+.0f} MW).")
print("The proportional-dispatch PF is descriptive only; Section 9a performs the dispatch-aware AC-OPF check.")

In [ ]:
# ---- Figure: train demand, adequacy, voltage envelope ----
fig,ax=plt.subplots(3,1,figsize=(12,10),sharex=True); hrs=np.arange(N)/3600.0
ax[0].plot(hrs,sysD,c="#c1121f",lw=0.6); ax[0].fill_between(hrs,sysD,color="#c1121f",alpha=0.2)
ax[0].set(ylabel="MW",title="(a) Total train (traction) demand over 24 h  [regen NOT returned to grid]"); ax[0].grid(alpha=0.3)
ax[1].plot(tsh,tot,c="#333"); ax[1].axhline(SUMPMAX,ls="--",c="r",label=f"gen capacity {SUMPMAX:.0f} MW")
ax[1].fill_between(tsh,SUMPMAX,tot,where=tot>SUMPMAX,color="r",alpha=0.3,label="deficit -> needs investment")
ax[1].set(ylabel="MW",title="(b) Total system load (base + traction) vs generation capacity"); ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)
ax[2].plot(tsh,vmx,c="#2a9d8f",lw=0.8,label="max bus V"); ax[2].plot(tsh,vmean,c="#1f77b4",lw=0.8,label="mean bus V")
ax[2].plot(tsh,vmn,c="#c1121f",lw=0.8,label="min bus V")
ax[2].axhline(Vmin_lim,ls=":",c="0.4"); ax[2].axhline(Vmax_lim,ls=":",c="0.4")
ax[2].set(xlabel="hour of day",ylabel="voltage (pu)",title="(c) Sampled transmission bus-voltage envelope (limits 0.94-1.06)")
ax[2].set_ylim(0.55,1.1); ax[2].set_xlim(0,24); ax[2].legend(fontsize=8,ncol=3); ax[2].grid(alpha=0.3)
plt.tight_layout(); plt.savefig("fig4_gridpf.pdf"); plt.savefig("fig4_gridpf.png"); plt.show()

### 9a. Dispatch-aware AC-OPF diagnostic

At the same hourly/peak sample set, AC OPF first enforces the required 0.94–1.06 pu voltage band.
When that solve does not return a feasible point, a second diagnostic solve relaxes only the lower
voltage bound to 0.80 pu. A relaxed-only solution indicates that the local nonlinear solver found an
operating point outside the required voltage band; failure of both solves remains computationally
unresolved and is not labelled as a proof of physical infeasibility.

In [ ]:
from pypower.api import runopf
opt_opf = ppoption(VERBOSE=0, OUT_ALL=0, OPF_ALG=560, OPF_VIOLATION=1e-7)

def opf_at(ti, vlo):
    load = base_vec.copy(); load[cbrows] += Dbus[:, int(ti)]
    bt = mpc_split["bus"].copy(); bt[:,2] = load
    bt[:,11] = Vmax_lim; bt[:,12] = float(vlo)
    case_opf = {"version":"2","baseMVA":baseMVA,"bus":bt,"gen":gen0_full.copy(),
                "branch":mpc_split["branch"].copy(),"gencost":gencost0_full}
    try:
        result = runopf(case_opf, opt_opf)
    except Exception:
        return dict(ok=False, vmin=np.nan, vmean=np.nan, vmax=np.nan, load=float(load.sum()))
    if not bool(result.get("success", False)):
        return dict(ok=False, vmin=np.nan, vmean=np.nan, vmax=np.nan, load=float(load.sum()))
    Vm = np.asarray(result["bus"][:,7], float)
    return dict(ok=True, vmin=float(Vm.min()), vmean=float(Vm.mean()),
                vmax=float(Vm.max()), load=float(load.sum()))

osamp = list(samples)
O_required, O_relaxed = [], []
for ti in osamp:
    req = opf_at(ti, Vmin_lim)
    rel = req if req["ok"] else opf_at(ti, 0.80)
    O_required.append(req); O_relaxed.append(rel)
otsh = np.asarray(osamp, float)/3600.0
oreq_feas = np.asarray([x["ok"] for x in O_required], bool)
orel_feas = np.asarray([x["ok"] for x in O_relaxed], bool)
orelaxed_only = (~oreq_feas) & orel_feas
ounresolved = ~orel_feas
ovmn_req = np.asarray([x["vmin"] for x in O_required], float)
ovmn_rel = np.asarray([x["vmin"] for x in O_relaxed], float)
oload = np.asarray([x["load"] for x in O_required], float)

print("---- sampled AC-OPF diagnostic ----")
print(f"Required-band feasible: {int(oreq_feas.sum())}/{len(osamp)} sampled points")
print(f"Relaxed-lower-bound only: {int(orelaxed_only.sum())}/{len(osamp)} sampled points")
print(f"Unresolved by both local solves: {int(ounresolved.sum())}/{len(osamp)} sampled points")
if oreq_feas.any():
    print(f"Required-band feasible voltage floor: min {np.nanmin(ovmn_req):.3f} pu")
if orelaxed_only.any():
    print(f"Relaxed-only voltage floor: min {np.nanmin(ovmn_rel[orelaxed_only]):.3f} pu")
print(f"One-second nameplate-capacity deficit duration remains {gen_deficit_h:.2f} h/day.")
print("These sampled diagnostics motivate the expansion planning of the paper (Figs. 4 and 5).")

In [ ]:
# ---- Figure: proportional-dispatch PF and sampled AC-OPF voltage floors ----
fig,ax=plt.subplots(figsize=(12,4.6))
ax.plot(tsh, vmn, c="#c1121f", lw=0.7, alpha=0.8, label="min bus V, proportional-dispatch PF")
if oreq_feas.any():
    ax.plot(otsh[oreq_feas], ovmn_req[oreq_feas], "o-", c="#1f77b4", ms=3, lw=1.0,
            label="AC OPF feasible within 0.94–1.06 pu")
if orelaxed_only.any():
    ax.plot(otsh[orelaxed_only], ovmn_rel[orelaxed_only], "s", c="#f28e2b", ms=4,
            label="AC OPF found only with 0.80 pu lower bound")
if ounresolved.any():
    for x in otsh[ounresolved]: ax.axvline(x, color="0.85", lw=2, zorder=0)
    ax.plot([],[]," ",label="grey line: both sampled local OPFs unresolved")
ax.axhline(Vmin_lim, ls=":", c="0.3")
if np.isfinite(base_vmin):
    ax.axhline(base_vmin, ls="--", c="0.6", label=f"no-traction PF floor {base_vmin:.3f}")
ax.set(xlim=(0,24), ylim=(0.55,1.08), xlabel="hour of day", ylabel="minimum bus voltage (pu)",
       title="Sampled voltage diagnostic before expansion")
ax.legend(fontsize=8, loc="lower center", ncol=2); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("fig5_opf.pdf"); plt.savefig("fig5_opf.png"); plt.show()

## 10. Planning inputs. Design-day envelopes and seeded timetable-jitter days

The planning stages of the paper do not read the one-second signature directly. They read (i) the nominal day reduced
to 15-minute envelopes, with a chronological energy layer (interval means), a coincident-peak security layer (all
substations sampled at the same second, the system-demand peak of each interval), coherent regeneration snapshots and
signed 10-second reserve products, and (ii) independent jittered timetable days produced by a seeded generator. Both
are built here exactly as in the paper.


### 10.1 Nominal design-day envelopes

In [ ]:
# ---- design-day data prep: canonical TSS signature -> 15-minute planning envelopes ----
case = {"version":"2","baseMVA":baseMVA,"bus":mpc_split["bus"],"gen":gen0_full,
        "branch":mpc_split["branch"],"gencost":gencost0_full}
busids = mpc_split["bus"][:,0].astype(int); NBm=len(busids); bidx={int(b):i for i,b in enumerate(busids)}

# Dbus/Rbus use the exact same physical-TSS ordering and section allocation exported in Section 7.
Dbus = np.asarray(Dt, float).copy()
Rbus = np.asarray(Rt, float).copy()
assert Dbus.shape == Rbus.shape == (len(coupling), N)
assert np.allclose(Dbus.sum(0), D.sum(0), atol=1e-4)
assert np.allclose(Rbus.sum(0), R.sum(0), atol=1e-4)

# reduce to design-day 15-min envelopes (T=96): mean / peak / ramp, coincident peak, regen
STEP=900; T=N//STEP
def _reduce(X):
    Xb=X[:,:T*STEP].reshape(X.shape[0],T,STEP)
    return Xb.mean(2), Xb.max(2), np.abs(np.diff(Xb[:,:,::10],axis=2)).max(2)
Dbar,Dpk,Dramp=_reduce(Dbus); Rbar,Rpk,_=_reduce(Rbus)
_sys=Dbus.sum(0); Dcoin=np.zeros((len(coupling),T))
for t in range(T):
    w=slice(t*STEP,(t+1)*STEP); kk=int(_sys[w].argmax()); Dcoin[:,t]=Dbus[:,t*STEP+kk]
# Rolling signed 10-s ramps; windows may straddle 15-min boundaries.
_roll=np.roll(_sys, -10)-_sys
RampUp=np.array([np.maximum(_roll[t*STEP:(t+1)*STEP],0).max() for t in range(T)])
RampDn=np.array([np.maximum(-_roll[t*STEP:(t+1)*STEP],0).max() for t in range(T)])
RampSys=np.maximum(RampUp,RampDn)
# Same-second regeneration at the coincident demand peak.
_pksec=np.array([t*STEP+int(_sys[t*STEP:(t+1)*STEP].argmax()) for t in range(T)])
Rcoin=np.stack([Rbus[:,int(sx)] for sx in _pksec],axis=1)
print(f"10-s reserve products: UP {RampUp.max():.1f} MW | DOWN {RampDn.max():.1f} MW")
print(f"temporal split: energy layer uses Dbar ({Dbar.sum()*0.25:.0f} MWh/day = simulated {(_sys.sum()/3600):.0f}); "
      f"security layer uses Dcoin (peak {Dcoin.sum(0).max():.0f} MW vs mean-layer {Dbar.sum(0).max():.0f} MW)")
print(f"Envelopes: {len(coupling)} physical TSSs x {T} steps (15 min); "
      f"demand mean-peak {Dbar.max():.0f} MW, 1-s peak {Dpk.max():.0f} MW, regen peak {Rpk.max():.0f} MW")

# base electrical data
base_Pd=case["bus"][:,2].copy(); base_Qd=case["bus"][:,3].copy()
GEN_BUS=case["gen"][:,0].astype(int)
Pmax=case["gen"][:,8]; Pmin=case["gen"][:,9]; Qmax=case["gen"][:,3]; Qmin=case["gen"][:,4]; SUMPMAX=float(Pmax.sum())

# Candidate generation sites of the paper (fixed ex ante). Storage candidates are the matched
# grid-side / rail-side pair at every traction substation.
cand_gen=list(new_gen_candidates)   # [5, 12, 17, 18, 51, 26, 15, 53], Section 2

d=dict(busids=list(busids), NB=NBm, T=T, coupling_bus=cb_ids,
       Dbar=Dbar,Dpk=Dpk,Dcoin=Dcoin,Dramp=Dramp,RampSys=RampSys,RampUp=RampUp,RampDn=RampDn,
       Rbar=Rbar,Rpk=Rpk,Rcoin=Rcoin,
       base_Pd=base_Pd,base_Qd=base_Qd,GEN_BUS=GEN_BUS,Pmax=Pmax,Pmin=Pmin,Qmax=Qmax,Qmin=Qmin,
       baseMVA=baseMVA,gen=case["gen"],gencost=gencost0_full,
       branch_split=mpc_split["branch"],bus_split=mpc_split["bus"],cand_gen=cand_gen,
       section_to_tss=np.asarray(SECTION_TO_TSS,int), section_to_bus=np.asarray(SECTION_TO_BUS,int))
print(f"Grid {NBm} buses, existing capacity {SUMPMAX:.0f} MW; generation candidates {len(cand_gen)}; "
      f"storage candidates: one grid-side and one rail-side option at each of the {len(cb_ids)} TSS.")

### 10.2 Seeded two-regime timetable-jitter generator

Operational uncertainty is applied to event times between fixed electrical drive cycles: origin delays, dwell
deviations, section-time disturbances, network-day and corridor-day common shocks, and bounded delay recovery. Every
departure respects the scheduled time (no early departure), the minimum headway and the minimum dwell, and prior-day
runs carry over midnight so the day is cyclic. Two stylized regimes, J1 (mild) and J2 (severe), are declared as
explicit bounds. The seed convention of the paper is as follows.

* training family (used by the paper's guidance loop): `20000 + k` for J1 and `21000 + k` for J2, `k = 0..24`;
* validation family of the paper (never used in planning): `40000 + k` for J1 and `41000 + k` for J2, `k = 0..49`;
* further independent families for the seed-sensitivity study: `35000`, `45000`, `50000` with the same offsets.

`sim_jitter_day(rng, **JIT_REGIMES[regime])` returns the per-TSS one-second demand and regeneration of one day and
`reduce_day` reduces it to the 15-minute envelopes of Section 10.1.


In [ ]:
# ---- Section 10.2: seeded timetable-jitter generator and the training family ----
import time

K_TRAIN = 25               # training days per regime in the paper
TRAIN_SEED0 = 20_000       # training seeds: 20000+k (J1), 21000+k (J2); 40000+ reserved for validation days

# Stylized operating regimes. Bounds are declared assumptions, not fitted distributions. Their
# minutes-scale magnitudes are consistent with the delay variability of empirical HSR records [25].
# dep_j       : individual origin delay U[0, dep_j]
# dwell_lo/hi : stop-specific dwell deviation [s]
# run_j       : signed section-time disturbance [s] applied between fixed electrical drive cycles
# day_j       : network-day common positive shock [s]
# corr_j      : corridor-day common positive shock [s]
# recovery_*  : bounded fraction/cap of accumulated delay recoverable before the next leg
JIT_REGIMES = {
    "J1": dict(dep_j=60,  dwell_lo=-15, dwell_hi=45, run_j=12,
               day_j=15, corr_j=10, recovery_frac=0.35, recovery_cap=20),
    "J2": dict(dep_j=120, dwell_lo=-30, dwell_hi=90, run_j=25,
               day_j=35, corr_j=25, recovery_frac=0.50, recovery_cap=45),
}
MIN_DWELL_S = 60


# ---------- instrumented drive cycles ----------
def _dc_segments(stations_km):
    p, m, b = drive_cycle(stations_km)
    segs = []; i = 0; nd = int(DWELL); aux = P_AUX / 1e6
    for k in range(len(stations_km) - 1):
        if k < len(stations_km) - 2:
            j = i
            while not (np.all(m[j:j+nd] == aux) and np.all(b[j:j+nd] == 0.0)
                       and (j + nd >= len(m) or m[j+nd] != aux or b[j+nd] != 0.0)):
                j += 1
            segs.append(("leg", i, j)); segs.append(("dwell", j, j + nd)); i = j + nd
        else:
            segs.append(("leg", i, len(m)))
    assert segs[0][1] == 0 and segs[-1][2] == len(m)
    for a2, b2 in zip(segs, segs[1:]): assert a2[2] == b2[1]
    return p, m, b, segs


CYC_SCN = {}
for _cd in corr:
    _p0, _m0, _b0, _sg = _dc_segments(_cd["stations"])
    _pp, _mm, _bb = drive_cycle(_cd["stations"])
    assert np.array_equal(_m0, _mm) and np.array_equal(_b0, _bb), "instrumented cycle diverged"
    CYC_SCN[_cd["name"]] = (_p0, _m0, _b0, _sg)
_nsecR = sum(cd["nsec"] for cd in corr); _NCBr = len(cb_ids); _NR = 86400
_sec2jR = np.asarray(SECTION_TO_TSS, dtype=int).copy()
assert len(_sec2jR) == _nsecR and (_sec2jR >= 0).all()


def _add_trace(M, B, sec, start, m, b):
    """Add one fixed electrical leg to the retained 24 h window; return whether it contributed."""
    start = int(start); n = len(m)
    lo = max(0, -start); hi = min(n, _NR - start)
    if hi <= lo: return False
    tt = start + np.arange(lo, hi, dtype=np.int64)
    np.add.at(M, (sec[lo:hi], tt), m[lo:hi])
    np.add.at(B, (sec[lo:hi], tt), b[lo:hi])
    return True


def _add_aux(M, section, start, end):
    """Auxiliary/hotel power while dwelling or held by headway control."""
    lo = max(0, int(start)); hi = min(_NR, int(end))
    if hi <= lo: return False
    M[int(section), lo:hi] += np.float32(P_AUX/1e6)
    return True


def sim_jitter_day(rng, dep_j, dwell_lo, dwell_hi, run_j=0, day_j=0, corr_j=0,
                   recovery_frac=0.0, recovery_cap=0, return_diag=False):
    """Generate one cyclic, headway-feasible operating day at 1 s resolution.

    The electrical drive cycle on each leg remains the calibrated fixed profile. Operational
    uncertainty changes event times between legs. Every leg departure is constrained by:
      actual >= scheduled                       (no early passenger departure)
      actual[n] >= actual[n-1] + MIN_HEADWAY_S (no overtaking / minimum separation)
      actual >= previous arrival + MIN_DWELL_S  (no overlapping leg and dwell traces)
    """
    M = np.zeros((_nsecR, _NR), np.float32)
    B = np.zeros((_nsecR, _NR), np.float32)
    _days = tuple(int(x) for x in CYCLIC_DAY_OFFSETS)
    _day_shock = {dd: int(rng.integers(0, int(day_j)+1)) if day_j > 0 else 0 for dd in _days}
    _corr_shock = {(dd, ci): int(rng.integers(0, int(corr_j)+1)) if corr_j > 0 else 0
                   for dd in _days for ci in range(len(corr))}
    diag = dict(min_headway_s=float("inf"), headway_holds=0, no_early_holds=0,
                early_departure_violations=0, headway_violations=0,
                prior_day_carryover_runs=0, current_day_runs=0)

    for ci, cd in enumerate(corr):
        p0, m0, b0, segs = CYC_SCN[cd["name"]]
        legs = [(a2, b2) for kind, a2, b2 in segs if kind == "leg"]
        for dr in (0, 1):
            p = p0 if dr == 0 else (cd["L"] - p0)
            sec_full = cd["soff"] + np.clip((p / cd["seclen"]).astype(int), 0, cd["nsec"] - 1)
            prev_start = np.full(len(legs), -10**12, dtype=np.int64)
            schedules = [(dd, t0) for dd in _days
                         for t0 in departures(stream_phase_s(ci, dr), dd)]
            schedules.sort(key=lambda x: x[1])

            for dd, t0 in schedules:
                if dd == 0: diag["current_day_runs"] += 1
                origin_delay = (_day_shock[dd] + _corr_shock[(dd, ci)]
                                + (int(rng.integers(0, int(dep_j)+1)) if dep_j > 0 else 0))
                contributed = False
                prev_actual = prev_sched = prev_end = prev_sec_end = None

                for li, (a2, b2) in enumerate(legs):
                    sched = int(t0 + a2)
                    if li == 0:
                        raw = int(t0 + origin_delay)
                    else:
                        dwell_dev = int(rng.integers(int(dwell_lo), int(dwell_hi)+1))
                        dwell_s = max(MIN_DWELL_S, int(round(DWELL)) + dwell_dev)
                        run_dev = int(rng.integers(-int(run_j), int(run_j)+1)) if run_j > 0 else 0
                        delay_now = max(int(prev_actual - prev_sched), 0)
                        recovery = min(int(recovery_cap), int(np.floor(float(recovery_frac)*delay_now)))
                        raw = max(int(prev_end + MIN_DWELL_S),
                                  int(prev_end + dwell_s + run_dev - recovery))

                    if raw < sched:
                        diag["no_early_holds"] += 1
                    no_early = max(raw, sched)
                    hw_floor = int(prev_start[li] + MIN_HEADWAY_S)
                    actual = max(no_early, hw_floor)
                    if actual > no_early:
                        diag["headway_holds"] += 1
                    if actual < sched:
                        diag["early_departure_violations"] += 1
                    if prev_start[li] > -10**11:
                        gap = int(actual - prev_start[li])
                        diag["min_headway_s"] = min(diag["min_headway_s"], gap)
                        if gap < MIN_HEADWAY_S:
                            diag["headway_violations"] += 1

                    if li > 0:
                        contributed |= _add_aux(M, prev_sec_end, prev_end, actual)
                    leg_sec = sec_full[a2:b2]
                    contributed |= _add_trace(M, B, leg_sec, actual, m0[a2:b2], b0[a2:b2])
                    prev_start[li] = actual
                    prev_actual, prev_sched = actual, sched
                    prev_end = int(actual + (b2-a2))
                    prev_sec_end = int(leg_sec[-1])

                if dd < 0 and contributed:
                    diag["prior_day_carryover_runs"] += 1

    if not np.isfinite(diag["min_headway_s"]): diag["min_headway_s"] = None
    assert diag["early_departure_violations"] == 0, diag
    assert diag["headway_violations"] == 0, diag
    if diag["min_headway_s"] is not None:
        assert diag["min_headway_s"] >= MIN_HEADWAY_S, diag

    ru = np.minimum(M, B); D = M - ru; R = B - ru
    Db = np.zeros((_NCBr, _NR), np.float32); Rb = np.zeros((_NCBr, _NR), np.float32)
    for j in range(_NCBr):
        rows = np.where(_sec2jR == j)[0]
        if len(rows):
            Db[j] = D[rows].sum(0); Rb[j] = R[rows].sum(0)
    return (Db, Rb, diag) if return_diag else (Db, Rb)


def reduce_day(Db, Rb):
    """Dual-temporal reduction with coherent demand-peak AND regeneration-peak snapshots.

    energy   : Dbar/Rbar are interval means;
    draw     : Dcoin/Rcoin are sampled together at the system-demand peak second;
    regen    : Dregen/Rregen are sampled together at the system-surplus-regen peak second;
    reserve  : RampUp/RampDn are signed rolling 10-s demand excursions.
    """
    STEP = 900; T = _NR // STEP
    _sysD = Db.sum(0); _sysR = Rb.sum(0)
    secD = [t*STEP + int(_sysD[t*STEP:(t+1)*STEP].argmax()) for t in range(T)]
    secR = [t*STEP + int(_sysR[t*STEP:(t+1)*STEP].argmax()) for t in range(T)]
    Dc = np.stack([Db[:, int(x)] for x in secD], axis=1).astype(np.float32)
    Rc = np.stack([Rb[:, int(x)] for x in secD], axis=1).astype(np.float32)
    Dr = np.stack([Db[:, int(x)] for x in secR], axis=1).astype(np.float32)
    Rr = np.stack([Rb[:, int(x)] for x in secR], axis=1).astype(np.float32)
    Dbar_d = Db[:, :T*STEP].reshape(_NCBr, T, STEP).mean(2)
    Rbar_d = Rb[:, :T*STEP].reshape(_NCBr, T, STEP).mean(2)
    _roll = np.roll(_sysD, -10) - _sysD
    Rup = np.array([np.maximum(_roll[t*STEP:(t+1)*STEP], 0).max() for t in range(T)])
    Rdn = np.array([np.maximum(-_roll[t*STEP:(t+1)*STEP], 0).max() for t in range(T)])
    return dict(Dcoin=Dc, Dbar=Dbar_d, Rbar=Rbar_d, Rcoin=Rc,
                Dregen=Dr, Rregen=Rr,
                RampUp=Rup, RampDn=Rdn, Ramp=np.maximum(Rup, Rdn),
                pk=float(_sysD.max()), sec_draw=np.asarray(secD), sec_regen=np.asarray(secR))


# ---------- generate the training family of the paper (25 days per regime) ----------
TRAIN_DAYS = []
TRAIN_TIMING_DIAG = []
_t0 = time.time()
for _ri, (_jn, _jk) in enumerate(JIT_REGIMES.items()):
    for _k in range(K_TRAIN):
        rng = np.random.default_rng(TRAIN_SEED0 + _ri * 1000 + _k)
        Db, Rb, _dg = sim_jitter_day(rng, return_diag=True, **_jk)
        TRAIN_TIMING_DIAG.append(dict(regime=_jn, **_dg))
        TRAIN_DAYS.append(dict(regime=_jn, timing=_dg, **reduce_day(Db, Rb)))

_min_hw = min(x["min_headway_s"] for x in TRAIN_TIMING_DIAG if x["min_headway_s"] is not None)
assert _min_hw >= MIN_HEADWAY_S
assert sum(x["early_departure_violations"] for x in TRAIN_TIMING_DIAG) == 0
assert sum(x["headway_violations"] for x in TRAIN_TIMING_DIAG) == 0
print(f"Training family: {len(TRAIN_DAYS)} feasible jittered days ({K_TRAIN} x {list(JIT_REGIMES)}) "
      f"in {time.time()-_t0:.0f}s | training seeds {TRAIN_SEED0}+ (J1) / {TRAIN_SEED0+1000}+ (J2); validation seeds 40000+ / 41000+")
print(f"timetable checks: minimum realized separation {_min_hw:.0f}s | "
      f"headway holds {sum(x['headway_holds'] for x in TRAIN_TIMING_DIAG):,} | "
      f"no-early holds {sum(x['no_early_holds'] for x in TRAIN_TIMING_DIAG):,} | "
      f"prior-day carry-over runs {sum(x['prior_day_carryover_runs'] for x in TRAIN_TIMING_DIAG):,}")

# ---------- publish the nominal day with its coherent regeneration-peak snapshots ---------------
_nom_sysR = Rbus.sum(0)
_nom_secR = np.array([t*900 + int(_nom_sysR[t*900:(t+1)*900].argmax()) for t in range(d["T"])])
_nom_Dregen = np.stack([Dbus[:, int(s)] for s in _nom_secR], axis=1).astype(np.float32)
_nom_Rregen = np.stack([Rbus[:, int(s)] for s in _nom_secR], axis=1).astype(np.float32)
d["Dregen"] = _nom_Dregen
d["Rregen"] = _nom_Rregen
NOMINAL_DAY = dict(
    regime="NOMINAL",
    Dcoin=np.asarray(d["Dcoin"], float), Dbar=np.asarray(d["Dbar"], float),
    Rcoin=np.asarray(d["Rcoin"], float), Rbar=np.asarray(d["Rbar"], float),
    Dregen=_nom_Dregen, Rregen=_nom_Rregen,
    RampUp=np.asarray(d["RampUp"], float), RampDn=np.asarray(d["RampDn"], float),
    Ramp=np.maximum(np.asarray(d["RampUp"], float), np.asarray(d["RampDn"], float)),
    pk=float(np.asarray(d["Dcoin"], float).sum(0).max()),
)

print("\nPLANNING DATA: cyclic nominal day (design basis) and the training family (jittered days used by the guidance loop).")
print(f"  nominal traction peak {NOMINAL_DAY['pk']:.1f} MW | "
      f"energy {NOMINAL_DAY['Dbar'].sum()*0.25:.1f} MWh/day | "
      f"10-s up/down {NOMINAL_DAY['RampUp'].max():.1f}/{NOMINAL_DAY['RampDn'].max():.1f} MW")
for _rn in JIT_REGIMES:
    _dd = [x for x in TRAIN_DAYS if x["regime"] == _rn]
    _pk = np.array([x["Dcoin"].sum(0).max() for x in _dd], float)
    _en = np.array([x["Dbar"].sum()*0.25 for x in _dd], float)
    print(f"  {_rn} training days: peak p05/p50/p95 "
          f"{np.quantile(_pk,0.05):.1f}/{np.median(_pk):.1f}/{np.quantile(_pk,0.95):.1f} MW | "
          f"energy p05/p50/p95 {np.quantile(_en,0.05):.1f}/{np.median(_en):.1f}/{np.quantile(_en,0.95):.1f} MWh")


### 10.3 Validation families (optional export)

The paper judges every plan on 100 independent days, 50 per regime, drawn from the seed family `40000`. The cell
below regenerates any family from its seed and stores the reduced envelopes of every day in one file, so the paper's
validation days can be inspected or re-used without the optimization pipeline. Set `EXPORT_FAMILIES` to the families
you want (each takes one to two minutes).


In [ ]:
EXPORT_FAMILIES = [40000]          # e.g. [40000, 35000, 45000, 50000]; [] skips the export
DAYS_PER_REGIME = 50               # the paper uses 50 J1 + 50 J2 days per family

def day_seed(family, regime, k):
    """Seed convention of the paper: family + 1000*regime_index + k, regime index 0 for J1 and 1 for J2."""
    return int(family) + 1000 * list(JIT_REGIMES).index(regime) + int(k)

def generate_family(family, days_per_regime=DAYS_PER_REGIME):
    out = {"family": int(family), "regime": [], "seed": [], "min_headway_s": [], "peak_MW": [], "energy_MWh": []}
    arrays = {k: [] for k in ("Dbar", "Dcoin", "Rbar", "Rcoin", "Dregen", "Rregen", "RampUp", "RampDn")}
    t0 = time.time()
    for rn, rk in JIT_REGIMES.items():
        for k in range(days_per_regime):
            sd = day_seed(family, rn, k)
            Db, Rb, dg = sim_jitter_day(np.random.default_rng(sd), return_diag=True, **rk)
            red = reduce_day(Db, Rb)
            for key in arrays: arrays[key].append(np.asarray(red[key], np.float32))
            out["regime"].append(rn); out["seed"].append(sd); out["min_headway_s"].append(dg["min_headway_s"])
            out["peak_MW"].append(float(red["Dcoin"].sum(0).max())); out["energy_MWh"].append(float(red["Dbar"].sum() * 0.25))
    fname = f"hsr_days_family{int(family)}.npz"
    np.savez_compressed(fname, tss_bus=np.asarray(cb_ids, int), step_s=900,
                        regime=np.asarray(out["regime"]), seed=np.asarray(out["seed"], int),
                        peak_MW=np.asarray(out["peak_MW"]), energy_MWh=np.asarray(out["energy_MWh"]),
                        **{k: np.stack(v) for k, v in arrays.items()})
    pk = np.asarray(out["peak_MW"]); en = np.asarray(out["energy_MWh"])
    print(f"family {family}: {len(out['seed'])} days -> {fname} in {time.time()-t0:.0f}s | "
          f"peak p05/p50/p95 {np.quantile(pk,.05):.1f}/{np.median(pk):.1f}/{np.quantile(pk,.95):.1f} MW | "
          f"energy p05/p50/p95 {np.quantile(en,.05):.1f}/{np.median(en):.1f}/{np.quantile(en,.95):.1f} MWh")
    return fname

FAMILY_FILES = [generate_family(f) for f in EXPORT_FAMILIES]
# The training family of Section 10.2 in the same format (arrays of 50 days: 25 J1 then 25 J2).
np.savez_compressed("hsr_days_training.npz", tss_bus=np.asarray(cb_ids, int), step_s=900,
                    regime=np.asarray([x["regime"] for x in TRAIN_DAYS]),
                    seed=np.asarray([TRAIN_SEED0 + 1000 * list(JIT_REGIMES).index(x["regime"]) + i % K_TRAIN for i, x in enumerate(TRAIN_DAYS)], int),
                    **{k: np.stack([np.asarray(x[k], np.float32) for x in TRAIN_DAYS]) for k in ("Dbar", "Dcoin", "Rbar", "Rcoin", "Dregen", "Rregen", "RampUp", "RampDn")})
print("training family written to hsr_days_training.npz")


## 11. Collect the benchmark files

Everything the paper's pipeline reads, in one folder. It holds the coupled MATPOWER case and its CSV matrices, the corridor
and substation geometry, the trainset and timetable parameters, the one-second nominal signature, the 15-minute
nominal envelopes, the seed conventions, and the day families exported above.


In [ ]:
import json, os, shutil
OUT_DIR = "benchmark"; os.makedirs(OUT_DIR, exist_ok=True)
for f in ["case57_hsr.m", "case57_hsr_bus.csv", "case57_hsr_branch.csv", "hsr_demand_signature.npz",
          "hsr_days_training.npz"] + list(FAMILY_FILES):
    if os.path.exists(f): shutil.copy(f, os.path.join(OUT_DIR, f))

# corridor and substation geometry (schematic units; U2KM converts to km)
geometry = dict(
    span_km=float(SPAN_KM), unit_to_km=float(U2KM),
    bus_xy={int(b): [float(x), float(y)] for b, (x, y) in pos57.items()},
    corridors={name: [int(b) for b in cb] for name, cb in corridors.items()},
    stations={f"ST{i+1:02d}": [float(p[0]), float(p[1])] for i, p in enumerate(station_xy)},
    tss=[dict(index=j, id=s["id"], bus=int(s["bus"]), type=s["type"], xy=[float(s["xy"][0]), float(s["xy"][1])],
              corridors=list(s["corridors"]), tie_km=float(s["tie_km"]),
              **({"split_line": [int(s["split"][0]), int(s["split"][1])], "fraction": float(s["frac"])} if s["type"] == "tap" else {}))
         for j, s in enumerate(tss)],
    tss_by_corridor={k: [int(j) for j in v] for k, v in TSS_BY_CORRIDOR.items()},
    section_to_tss=[int(x) for x in SECTION_TO_TSS], section_to_bus=[int(x) for x in SECTION_TO_BUS],
    section_length_km=float(SEC_LEN_KM), station_spacing_km=float(STATION_SPACING_KM), tss_spacing_km=float(TSS_SPACING_KM))
json.dump(geometry, open(os.path.join(OUT_DIR, "geometry.json"), "w"), indent=1)

# trainset, catenary, traction-transformer and timetable parameters (Section 7 values; Section II of the paper)
params = dict(
    trainset=dict(mass_kg=MASS, effective_mass_kg=M_EFF, wheel_power_W=P_WHEEL, starting_effort_N=F_START,
                  v_max_mps=V_MAX_T, service_brake_mps2=A_BRAKE, regen_cap_W=P_REGEN_MAX, eta_motoring=ETA_MOT,
                  eta_regen=ETA_REG, auxiliary_W=P_AUX, davis_A=float(A_D), davis_B=float(B_D), davis_C=float(C_D)),
    catenary=dict(z_eq_ohm_per_km=[Z_EQ.real, Z_EQ.imag], power_factor=PF, v_nom_V=V_NOM, v_max_V=V_MAX_CAT, v_min_V=V_MIN_CAT,
                  neutral_section_km=SEC_LEN_KM),
    traction_transformer=dict(x_pu=0.10, r_pu=0.010, rating_MVA=150.0, ratio=1.0,
                              note="one HV/25 kV transformer per TSS, feeding a 25 kV rail busbar; pu on the 100 MVA base (Section II-A)"),
    timetable=dict(dwell_s=DWELL, min_headway_s=MIN_HEADWAY_S, stream_phase_step_s=STREAM_PHASE_STEP_S,
                   cyclic_day_offsets=list(CYCLIC_DAY_OFFSETS), min_dwell_s=MIN_DWELL_S),
    jitter_regimes=JIT_REGIMES,
    seeds=dict(training=dict(family=TRAIN_SEED0, days_per_regime=K_TRAIN, rule="family + 1000*regime_index + k"),
               validation=dict(paper_family=40000, other_families=[35000, 45000, 50000], days_per_regime=50,
                               rule="family + 1000*regime_index + k (regime_index 0 = J1, 1 = J2)")))
json.dump(params, open(os.path.join(OUT_DIR, "parameters.json"), "w"), indent=1, default=float)

# nominal-day 15-minute envelopes (Section 10.1)
np.savez_compressed(os.path.join(OUT_DIR, "hsr_nominal_envelopes.npz"), tss_bus=np.asarray(cb_ids, int), step_s=900,
                    Dbar=d["Dbar"], Dcoin=d["Dcoin"], Dpk=d["Dpk"], Rbar=d["Rbar"], Rcoin=d["Rcoin"],
                    Dregen=d["Dregen"], Rregen=d["Rregen"], RampUp=d["RampUp"], RampDn=d["RampDn"],
                    base_Pd=d["base_Pd"], base_Qd=d["base_Qd"], busids=np.asarray(d["busids"], int), cand_gen=np.asarray(cand_gen, int))
print("benchmark files:")
for f in sorted(os.listdir(OUT_DIR)): print(f"  {f:36s} {os.path.getsize(os.path.join(OUT_DIR, f))/1e3:9.1f} kB")


## References

Numbers in square brackets in the code comments refer to the bibliography of the paper. The entries used in this
notebook are listed below with their numbers in the paper.

- **[1]** H. Hu, Y. Liu, Y. Li, Z. He, S. Gao, X. Zhu, and H. Tao, "Traction power systems for electrified railways: Evolution, state of the art, and future trends," *Railw. Eng. Sci.*, vol. 32, no. 1, pp. 1–19, Mar. 2024.
- **[4]** M. Khodaparastan, A. A. Mohamed, and W. Brandauer, "Recuperation of regenerative braking energy in electric rail transit systems," *IEEE Trans. Intell. Transp. Syst.*, vol. 20, no. 8, pp. 2831–2847, Aug. 2019.
- **[21]** R. D. Zimmerman, C. E. Murillo-Sánchez, and R. J. Thomas, "MATPOWER: Steady-state operations, planning, and analysis tools for power systems research and education," *IEEE Trans. Power Syst.*, vol. 26, no. 1, pp. 12–19, Feb. 2011.
- **[22]** *Railway applications – Supply voltages of traction systems*, CENELEC Standard EN 50163, 2004.
- **[23]** C. Lu, B. Zhang, and H. Zhao, "CR-Fuxing high-speed EMU series," *Front. Eng. Manag.*, vol. 10, no. 4, pp. 742–748, Dec. 2023.
- **[24]** B. P. Rochard and F. Schmid, "A review of methods to measure and calculate train resistances," *Proc. Inst. Mech. Eng. F, J. Rail Rapid Transit*, vol. 214, no. 4, pp. 185–199, Jul. 2000.
- **[25]** D. Zhang et al., "A high-speed railway network dataset from train operation records and weather data," *Sci. Data*, vol. 9, art. no. 244, May 2022.

**External data source.** The branch thermal ratings are taken from the PGLib-OPF case `pglib_opf_case57_ieee`
(v23.07).

- **[PGLib]** S. Babaeinejadsarookolaee et al., "The Power Grid Library for benchmarking AC optimal power flow algorithms," arXiv:1908.02788, 2019.

**Declared assumptions.** The following values are design choices of the benchmark and carry no cited source. They
are the 400 km geographic span, the corridor routing and waypoints, the 45 km station spacing, the 50 km target TSS
spacing and the 25 km snap distance, the 24 km neutral-section arm, the equivalent catenary impedance
0.04 + j0.10 ohm/km, the fixed 0.98 traction power factor, the traction-transformer data (10% impedance, X/R ≈ 10,
150 MVA), the rotational-inertia allowance, the drivetrain efficiencies, the service deceleration, the 120 s dwell,
the 6/10/15 min headways with a 180 s minimum separation, and the bounds of the two jitter regimes.
